# Sprint 0 — Specification, Scope and Acceptance Criteria

**Project title:** *Elevating Academic Integrity: Unveiling Fraud Detection in Online University Assessments using Wrapper Technique Feature Selection Method*

**Module:** Information Security & Data Mining · **Assessment type:** Individual, notebook-based

---

## 0.1 Research question

> To what extent can a **compact, interpretable, movement-derived feature set** — selected from a three-feature candidate pool by a **wrapper feature-selection method** — discriminate between the user labels present in the supplied three-point mouse-movement segment data, and which classifier configuration produces the most reliable and best-calibrated identification result?

This is an **exploratory multi-class identification exercise**. It is *not* evidence that the method is fit for real-world authentication, proctoring or fraud interdiction. Section 8 states the boundaries explicitly and no cell in this notebook asserts a deployment claim.

## 0.2 Development methodology (why this notebook is organised this way)

The development of this project follows an **Agile-style iterative loop governed by the Empirical Design Paradigm (Ralph, 2018)**, anchored by a single up-front specification phase. The reasoning, and the comparison with a plan-driven (Waterfall) approach, is documented in the project README; the short version is:

* The **requirements are already fixed and stable** (three named features, one target column, a prescribed evaluation contract). That removes the precondition that would normally make plan-driven work attractive.
* The **real uncertainty in this project is empirical**, not analytical: whether the delivered files contain the expected columns, how many classes exist, how separable the three features are, and which estimator and feature subset actually generalise. Empirical uncertainty is resolved by a short feedback loop, not by a longer plan.
* Each sprint therefore ends with a **runnable, inspectable artefact** (a data-quality summary, a figure, a metrics table) rather than a document describing work that has not yet been evidenced.

Every sprint below states a **goal**, a **user story**, and **acceptance criteria** — the criteria are the definition of "done" for that sprint, and they are fixed *before* the sprint runs, so that completion cannot be defined after the fact in a way that flatters the result.

## 0.3 Fixed specification (the contract this notebook implements)

| Item | Value | Rationale |
| --- | --- | --- |
| Input features | `avg_d`, `avg_tv`, `avg_a` | The three features fixed by the project README. Distance, speed and acceleration summarise a movement segment. |
| Target | `label` (multi-class user identity) | The class column supplied with the segment data. |
| Excluded inputs | raw `x*`, `y*`, `t*`, index columns | Absolute coordinates encode screen geometry; absolute timestamps and collection order encode acquisition order. Either can leak information that does not generalise to a new session. |
| Split | `train_test_split(test_size=0.20, stratify=y, random_state=42)` | Stratified, reproducible, and the held-out test portion is never used for any selection decision. |
| Primary metric | macro-averaged F1 (+ macro precision / recall, accuracy) | Multi-class task; macro-averaging weights every user equally instead of letting the majority class dominate. |
| Selection method | wrapper feature selection over the three candidate features | The project title mandates the wrapper technique. |
| Evaluation | nested cross-validation on the training portion; one final evaluation on the held-out test portion | Prevents the optimism that comes from selecting a model and a feature subset on the same data used to report performance. |

## 0.4 Integrity rules this notebook obeys

1. **No invented numbers.** No accuracy, F1 or confidence value is written into any Markdown cell. Every reported quantity is computed by a code cell and read from its output.
2. **No test-set tuning.** The test portion is touched exactly once, by the final evaluation cell, after all model and feature-subset decisions have been made on the training portion.
3. **No fabricated hyper-parameters.** Every estimator parameter used is verified programmatically against the installed scikit-learn signature (Section 4.3), and the scikit-learn default is printed alongside it.
4. **No silent data repair.** Missing columns, missing labels, non-numeric values and non-finite values are all reported with counts; nothing is imputed or relabelled without an explicit, printed notice.
5. **No causal claims from the exploratory figures.** The EDA describes the data; it does not establish why the classes separate.

## 0.5 Acceptance criteria for the whole deliverable

- [ ] The notebook executes top-to-bottom in a clean Google Colab runtime after the data path is supplied.
- [ ] The data-quality summary, class counts, train/test sizes and the number of rows removed during cleaning are all printed and can be checked by hand.
- [ ] All candidate models are compared under an identical split, identical folds and identical scoring, so the comparison is not contaminated by protocol differences.
- [ ] The wrapper result is compared against a no-selection baseline under the same protocol.
- [ ] Macro metrics and a confusion matrix are reported. A single false-positive rate is **not** reported, because this is not a binary authentication task.
- [ ] Overfitting is diagnosed with learning curves and train/validation gaps, not asserted.
- [ ] Every discussion cell contains a placeholder for the human author to complete after execution, and the placeholders are not filled with invented results.

In [ ]:
"""Sprint 0 (implementation) — environment, reproducibility constants, configuration.

Everything that a reviewer may want to change lives in this single cell.
"""

# ------------------------------------------------------------------ #
# 0.4  Optional dependency bootstrap
# ------------------------------------------------------------------ #
# Google Colab already ships numpy / pandas / matplotlib / seaborn /
# scikit-learn. The block below only runs if an import fails, so that the
# notebook is also usable in a plain Python environment.
import importlib
import subprocess
import sys

_REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
}

_missing = []
for _module, _package in _REQUIRED.items():
    try:
        importlib.import_module(_module)
    except ImportError:
        _missing.append(_package)

if _missing:
    print(f"Installing missing package(s): {', '.join(_missing)}")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
    except (subprocess.CalledProcessError, OSError) as error:
        print(f"Automatic installation failed ({error}).")
        print("Install the missing package(s) manually, for example:")
        print(f"    !pip install {' '.join(_missing)}")
        raise ImportError(f"Missing required package(s): {', '.join(_missing)}") from error
else:
    print("All required packages are already available.")

# ------------------------------------------------------------------ #
# Standard library
# ------------------------------------------------------------------ #
import itertools
import os
import platform
import time
import warnings
from pathlib import Path

# ------------------------------------------------------------------ #
# Third party
# ------------------------------------------------------------------ #
import matplotlib

# Render figures inline. In Google Colab and Jupyter this registers the inline
# backend so that every figure appears in the notebook output; in a plain
# Python interpreter it falls back to a non-interactive backend.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:  # pragma: no cover - only reached outside a notebook
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    learning_curve,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------------------------ #
# 0.1  Reproducibility
# ------------------------------------------------------------------ #
RANDOM_STATE = 42

# ------------------------------------------------------------------ #
# 0.2  The three selected features (fixed by the project README)
# ------------------------------------------------------------------ #
FEATURES = ["avg_d", "avg_tv", "avg_a"]
TARGET = "label"

# ------------------------------------------------------------------ #
# 0.3  Evaluation protocol
# ------------------------------------------------------------------ #
TEST_SIZE = 0.20

# Computational budget switch.
#   False (default) — the full nested search. This is the configuration the
#                     results in the report are based on.
#   True            — a quicker pass for a large dataset: three outer folds
#                     instead of five, a smaller SVC grid, and a cheaper forest
#                     inside the search. The mode in force is printed in the
#                     run manifest and in the Sprint 5 output, so a result can
#                     never be mistaken for a full-search result.
FAST_MODE = False

N_SPLITS_OUTER = 3 if FAST_MODE else 5  # folds of the selection loop (training portion only)
N_SPLITS_INNER = 3                     # folds used inside the wrapper search
N_SPLITS_FINAL = 3 if FAST_MODE else 5 # folds for the final selection on the full training portion
N_BOOTSTRAP = 2000                     # bootstrap resamples for the test-set confidence interval
CONF_LEVEL = 0.95

# Pre-registered tie-break order for the one-standard-error rule.
# Fixed BEFORE any result is observed, so the final model cannot be
# chosen retrospectively to suit the numbers.
TIE_BREAK_PRIORITY = [
    "Logistic Regression",
    "Random Forest",
    "RBF SVC",
    "K-Nearest Neighbours",
    "Gaussian Naive Bayes",
]

# ------------------------------------------------------------------ #
# 0.5  Display configuration
# ------------------------------------------------------------------ #
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

sns.set_theme(style="whitegrid", context="notebook")
PALETTE = "Set2"
FIGURE_DPI = 110


def banner(text: str) -> None:
    """Print a sprint banner so the console log reads like the notebook."""
    print("\n" + "=" * 78)
    print(text)
    print("=" * 78)


def show_versions() -> None:
    """Print the run manifest. A reviewer should be able to reproduce the run."""
    print("Run manifest")
    print("-" * 78)
    print(f"  python             : {platform.python_version()} ({platform.system()})")
    print(f"  numpy              : {np.__version__}")
    print(f"  pandas             : {pd.__version__}")
    print(f"  matplotlib         : {matplotlib.__version__}")
    print(f"  seaborn            : {sns.__version__}")
    print(f"  scikit-learn       : {sklearn.__version__}")
    print(f"  random_state       : {RANDOM_STATE}")
    print(f"  features           : {FEATURES}")
    print(f"  target             : {TARGET}")
    print(f"  split              : test_size={TEST_SIZE}, stratify=True")
    print(f"  CV (outer/inner)   : {N_SPLITS_OUTER} / {N_SPLITS_INNER} (StratifiedKFold, shuffled)")
    print(f"  bootstrap resamples: {N_BOOTSTRAP}")
    print(f"  FAST_MODE          : {FAST_MODE}"
          + ("  (reduced search — results are NOT from the full protocol)" if FAST_MODE else ""))


# ------------------------------------------------------------------ #
# 0.6  Column resolution helpers
# ------------------------------------------------------------------ #
def _normalise(name: str) -> str:
    """Lower-case and strip everything that is not a letter or a digit."""
    return "".join(ch for ch in str(name).lower() if ch.isalnum())


# Documented fallbacks, tried only when the exact name is absent.
FEATURE_ALIASES = {
    "avg_d": ["avgd", "aved", "avgedist", "avgdistance", "meaned", "meand", "d"],
    "avg_tv": ["avgtv", "tva", "avgtv", "tv", "avgvelocity", "avevelocity", "avgv"],
    "avg_a": ["avga", "a", "avgacc", "avgaccel", "avgacceleration", "meana"],
}
LABEL_ALIASES = ["label", "labels", "user", "userid", "username", "class", "target", "y"]


def resolve_column(frame: pd.DataFrame, wanted: str, aliases: list[str]) -> str | None:
    """Find the column in `frame` that corresponds to `wanted`."""
    if wanted in frame.columns:
        return wanted
    normalised = {_normalise(col): col for col in frame.columns}
    if _normalise(wanted) in normalised:
        return normalised[_normalise(wanted)]
    for alias in aliases:
        if _normalise(alias) in normalised:
            return normalised[_normalise(alias)]
    return None


def matching_features(frame: pd.DataFrame) -> list[str]:
    """Which of the three selected features can be resolved in this frame?"""
    found = []
    for feature in FEATURES:
        if resolve_column(frame, feature, FEATURE_ALIASES.get(feature, [])) is not None:
            found.append(feature)
    return found


def derive_label_from_filename(file_name: str) -> str | None:
    """Derive a user label from a file name such as User7_..._Segment.csv."""
    import re

    match = re.search(r"user[_\-\s]?(\d+)", str(file_name), flags=re.IGNORECASE)
    if match:
        return f"User{match.group(1)}"
    return None


def canonicalise_labels(series: pd.Series) -> pd.Series:
    """Put the labels in one consistent type.

    Numeric labels stay numeric (as integers when they are whole numbers);
    anything else is kept as text. This matters because the same user may
    arrive as the integer 3 in one file and as the string 'User3' in another.
    """
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().all():
        if np.allclose(numeric, numeric.round()):
            return numeric.astype(int)
        return numeric.astype(float)
    return series.astype(str)


def sort_labels(values) -> list:
    """Order labels the way a human reads them.

    Plain `sorted()` puts 'User10' between 'User1' and 'User2'. When the
    labels look like '<prefix><number>' the numeric part is used for the
    ordering; anything else falls back to ordinary alphabetical order.
    """
    import re

    def key(value):
        match = re.fullmatch(r"(.*?)(\d+)", str(value))
        if match:
            return (0, match.group(1), int(match.group(2)), str(value))
        return (1, "", 0, str(value))

    return sorted(list(values), key=key)


show_versions()

---
## Sprint 1 — Data access, loading and audit

**Goal.** Obtain the course segment data, load it without silently repairing anything, and produce a data-quality summary that a human can check by hand.

**User story.** *As a reviewer, I want to see exactly which files were read, how many rows and labels arrived, and what is missing, so that I can trust every later number.*

**Acceptance criteria for this sprint**

1. The list of CSV files actually found is printed before anything is loaded.
2. Loading either succeeds with all required columns present, or stops with an error that names the missing columns and the path that was tried.
3. A `label` column is never fabricated. If a file has no label column, the label is derived from the `UserN_...` file name and that derivation is printed loudly.
4. The sprint prints: file count, raw shape, class counts, duplicate-row count, and per-column missing-value counts.

**Data location.** The course files live in a shared Google Drive folder. In Colab, mount Drive once and point `DATA_DIR` at the folder; the loader also searches a few conventional locations automatically. Set `USE_SYNTHETIC_DEMO = True` only to smoke-test the pipeline when the real files are unavailable — results produced that way are labelled `synthetic` in every table and are **not** project results.

In [ ]:
"""Sprint 1 (implementation) — resolve the data directory, load, keep the source of every row.

Run order: configure DATA_DIR -> resolve -> list -> load -> audit.
"""

# ------------------------------------------------------------------ #
# 1.1  CONFIGURATION — edit this cell
# ------------------------------------------------------------------ #
# Absolute path to the folder that holds the course CSV files.
# On Colab after mounting Drive, a typical value is:
#     DATA_DIR = "/content/drive/MyDrive/<your shared folder name>"
DATA_DIR = "/content/drive/MyDrive/1hAYr22yo8wWvUvJF88JZfNQvgHlL7hxW"

# Set to True ONLY to smoke-test the pipeline with generated data.
# Every table produced in that mode is stamped "synthetic" and must not
# be reported as a result of this project.
USE_SYNTHETIC_DEMO = False

# Optional: candidate folders that are searched when DATA_DIR does not exist.
EXTRA_DATA_DIR_CANDIDATES = [
    "/content/drive/MyDrive",
    "/content/drive/MyDrive/1hAYr22yo8wWvUvJF88JZfNQvgHlL7hxW",
    "/content",
    "/mnt/data",
    ".",
]

# Files are expected to look like User1_Calib_3MouseMovement_MCQ_Segment.csv
FILE_GLOB = "**/*.csv"


# ------------------------------------------------------------------ #
# 1.2  Directory resolution
# ------------------------------------------------------------------ #
def _has_csv(directory: Path) -> bool:
    return any(directory.glob(FILE_GLOB))


def resolve_data_dir() -> Path | None:
    """Return the first configured directory that actually contains CSV files."""
    candidates = [DATA_DIR, *EXTRA_DATA_DIR_CANDIDATES]
    tried = []
    for candidate in candidates:
        if not candidate:
            continue
        path = Path(candidate).expanduser()
        tried.append(str(path))
        if path.is_dir() and _has_csv(path):
            return path.resolve()
    print("Could not locate a folder containing CSV files.")
    print("Locations tried:")
    for item in tried:
        print(f"  - {item}")
    print(
        "\nHow to fix this in Google Colab:\n"
        "  1. Runtime > Mount drive, then run:\n"
        "       from google.colab import drive\n"
        "       drive.mount('/content/drive')\n"
        "  2. Put the shared folder inside 'My Drive' (right-click > Add shortcut),\n"
        "  3. Set DATA_DIR above to the folder that contains the UserN_*.csv files,\n"
        "     e.g. DATA_DIR = '/content/drive/MyDrive/<folder name>'\n"
        "  4. Re-run this cell.\n"
        "Alternatively set USE_SYNTHETIC_DEMO = True to smoke-test the pipeline only."
    )
    return None


# ------------------------------------------------------------------ #
# 1.3  Synthetic smoke-test data (only used when the real files are absent)
# ------------------------------------------------------------------ #
def make_synthetic_demo(n_users: int = 10, n_segments: int = 400, seed: int = RANDOM_STATE) -> list[tuple[str, pd.DataFrame]]:
    """Generate schema-compatible frames so the pipeline can be exercised.

    One frame per user, with the same file-naming convention as the course
    data, so that the label-consistency audit is exercised by the same code
    path as on the real files. The per-user movement profiles are invented:
    any metric computed on them describes the generator, not the course data.
    """
    rng = np.random.default_rng(seed)
    frames: list[tuple[str, pd.DataFrame]] = []
    for user_index in range(1, n_users + 1):
        speed_level = 0.75 + 0.11 * user_index
        size_level = 0.70 + 0.13 * user_index
        # Per-segment draw factor, so that the three features are correlated
        # within a segment (distance -> velocity -> acceleration).
        draw = rng.normal(0.0, 0.55, size=n_segments)
        distance = np.exp(4.2 + size_level + draw) * rng.lognormal(0.0, 0.30, n_segments)
        velocity = np.exp(3.1 + speed_level + 0.75 * draw) * rng.lognormal(0.0, 0.25, n_segments)
        acceleration = np.exp(2.0 + 0.6 * speed_level + 0.55 * draw) * rng.lognormal(0.0, 0.35, n_segments)
        frames.append(
            (
                f"User{user_index}_Calib_3MouseMovement_MCQ_Segment.csv",
                pd.DataFrame(
                    {
                        "avg_d": distance,
                        "avg_tv": velocity,
                        "avg_a": acceleration,
                        "label": f"User{user_index}",
                    }
                ),
            )
        )
    return frames


# ------------------------------------------------------------------ #
# 1.3b  Tolerant reader
# ------------------------------------------------------------------ #
def read_course_csv(path: Path) -> pd.DataFrame:
    """Read one course CSV, tolerating non-numeric entries in numeric columns.

    pandas refuses to guess a dtype for a column that contains a stray
    string, and the failure is a hard error. Rather than crash the whole
    notebook on one awkward cell, the file is re-read with every column
    treated as text, and the cleaning step in Sprint 2 converts the three
    selected features to numeric and reports exactly what it dropped.
    """
    try:
        return pd.read_csv(path)
    except (ValueError, TypeError) as error:
        print(f"  NOTE: {path.name} could not be parsed with automatic dtypes.")
        print(f"        Reason: {error}")
        print("        Re-reading every column as text; Sprint 2 will convert the three")
        print("        selected features to numeric and report the rows it removes.")
        return pd.read_csv(path, dtype=str)


# ------------------------------------------------------------------ #
# 1.4  Load
# ------------------------------------------------------------------ #
banner("Sprint 1 — data access, loading and audit")

data_source = "course files"
source_files: list[str] = []
parts: list[pd.DataFrame] = []

if USE_SYNTHETIC_DEMO:
    banner("USE_SYNTHETIC_DEMO = True — SMOKE TEST ONLY, NOT PROJECT RESULTS")
    print("The cells below are running on generated data. Every metric they")
    print("produce describes the generator and must not be reported as a")
    print("result of this project. Set USE_SYNTHETIC_DEMO = False before")
    print("running the project for real.")
    demo_frames = make_synthetic_demo()
    parts = [frame for _, frame in demo_frames]
    source_files = [name for name, _ in demo_frames]
    data_source = "synthetic (smoke test)"
else:
    resolved_dir = resolve_data_dir()
    if resolved_dir is None:
        raise FileNotFoundError(
            "No data directory could be resolved. See the instructions printed above, "
            "or set USE_SYNTHETIC_DEMO = True for a pipeline smoke test."
        )
    files = sorted(p for p in resolved_dir.glob(FILE_GLOB) if p.is_file())
    print(f"Resolved DATA_DIR -> {resolved_dir}")
    print(f"CSV files found     -> {len(files)}")
    for path in files:
        print(f"  {path.name}  ({path.stat().st_size:,} bytes)")
    if not files:
        raise FileNotFoundError(f"No CSV files found under {resolved_dir}")
    skipped_files: list[tuple[str, str]] = []
    for path in files:
        frame = read_course_csv(path)
        # A file that does not contain the three selected features cannot
        # contribute to the experiment. It is skipped with a printed reason
        # rather than silently included or silently dropped.
        found = matching_features(frame)
        if len(found) < len(FEATURES):
            skipped_files.append(
                (path.name, f"contains only {found or 'none'} of the three selected features")
            )
            continue
        parts.append(frame)
        source_files.append(path.name)
    if skipped_files:
        print("\nFiles skipped (they do not contain the three selected features):")
        for name, reason in skipped_files:
            print(f"  {name}: {reason}")
        if not parts:
            raise FileNotFoundError(
                "None of the CSV files found contains the three selected features "
                f"{FEATURES}. Check DATA_DIR and the column names."
            )

# Keep the originating file on every row: needed to audit label consistency
# and to reason about possible session-level grouping.
for name, frame in zip(source_files, parts):
    frame["source_file"] = name

raw = pd.concat(parts, ignore_index=True)

banner(f"Raw frame loaded [{data_source}]")
print(f"Files read : {len(source_files)}")
print(f"Shape      : {raw.shape}")
print(f"Columns    : {list(raw.columns)}")
raw.head(8)

In [ ]:
"""Sprint 1 (continued) — label resolution, column validation, data-quality audit."""

# ------------------------------------------------------------------ #
# 1.5  Target column
# ------------------------------------------------------------------ #
label_column = resolve_column(raw, TARGET, LABEL_ALIASES)
label_derivation_notices: list[str] = []

if label_column is None:
    # No label column anywhere: derive it from the file names, one label per file.
    derived = raw["source_file"].map(derive_label_from_filename)
    if derived.isna().any():
        bad = sorted(raw.loc[derived.isna(), "source_file"].unique())
        raise KeyError(
            "No usable target column was found and the label could not be derived from the "
            f"file names of: {bad}. Expected a 'label' column (or a 'UserN_...' file name)."
        )
    raw[TARGET] = derived
    label_column = TARGET
    label_derivation_notices.append(
        "No label column was present in the source files. Labels were DERIVED from the "
        "file names (UserN_... -> 'UserN'). Verify this mapping before reporting results."
    )
else:
    if label_column != TARGET:
        raw = raw.rename(columns={label_column: TARGET})
        label_derivation_notices.append(
            f"The target column was named '{label_column}' in the source files and was renamed "
            f"to '{TARGET}' for consistency with the project specification."
        )
    # Files that do not carry a usable label value.
    for file_name, group in raw.groupby("source_file"):
        values = group[TARGET].dropna().unique()
        if len(values) == 0:
            derived = derive_label_from_filename(file_name)
            if derived is None:
                raise KeyError(
                    f"File '{file_name}' has no label values and no derivable user name. "
                    "Please supply the label column explicitly."
                )
            mask = (raw["source_file"] == file_name) & (raw[TARGET].isna())
            raw.loc[mask, TARGET] = derived
            label_derivation_notices.append(
                f"File '{file_name}' carried no label values; labels were derived from the "
                f"file name as '{derived}'."
            )

# One label space for the whole dataset: a label that arrives as the integer 3
# in one file and as the string 'User3' in another is the same user.
raw[TARGET] = canonicalise_labels(raw[TARGET])
if label_derivation_notices:
    label_derivation_notices.append(
        f"Labels were cast to a single type; the final label values are "
        f"{sorted(raw[TARGET].unique())[:12]}{' ...' if raw[TARGET].nunique() > 12 else ''}."
    )

banner("Label resolution")
for notice in label_derivation_notices:
    print(f"NOTICE: {notice}")
if not label_derivation_notices:
    print(f"Target column '{TARGET}' found in the source data; no labels were altered.")
print(f"Target column in use: {TARGET}")


# ------------------------------------------------------------------ #
# 1.6  Feature column validation
# ------------------------------------------------------------------ #
feature_source_columns: dict[str, str] = {}
missing_features: list[str] = []

for feature in FEATURES:
    matched = resolve_column(raw, feature, FEATURE_ALIASES.get(feature, []))
    if matched is None:
        missing_features.append(feature)
    else:
        feature_source_columns[feature] = matched

if missing_features:
    raise KeyError(
        f"The required feature column(s) {missing_features} are not present in the data.\n"
        f"Columns available: {list(raw.columns)}\n"
        "The project specification fixes the three features as "
        f"{FEATURES}. If the source files use different names, map them in the "
        "FEATURE_ALIASES dictionary in the configuration cell rather than renaming the data."
    )

for feature, source_column in feature_source_columns.items():
    if source_column != feature:
        raw = raw.rename(columns={source_column: feature})
        print(f"Mapped feature '{feature}' <- source column '{source_column}'")


# ------------------------------------------------------------------ #
# 1.7  Data-quality audit
# ------------------------------------------------------------------ #
banner("Sprint 1 — data-quality summary")

audit: dict[str, object] = {}
audit["data_source"] = data_source
audit["files_read"] = len(source_files)
audit["raw_shape"] = raw.shape
audit["n_columns"] = raw.shape[1]
audit["duplicate_rows"] = int(raw.drop(columns=["source_file"]).duplicated().sum())

feature_columns_for_audit = FEATURES + [TARGET]
missing_counts = raw[feature_columns_for_audit].isna().sum()
missing_counts = missing_counts[missing_counts > 0]

print(f"Data source        : {data_source}")
print(f"Files read         : {audit['files_read']}")
print(f"Raw shape          : {audit['raw_shape']}")
print(f"Duplicate rows     : {audit['duplicate_rows']:,}")
print(f"Selected features  : {FEATURES}  (resolved to {feature_source_columns})")
print(f"Target             : {TARGET}")
print()
print("Missing values per required column:")
if missing_counts.empty:
    print("  none")
else:
    print(missing_counts.to_string())

class_counts = raw[TARGET].value_counts().reindex(sort_labels(raw[TARGET].unique()))
print(f"\nDistinct labels    : {raw[TARGET].nunique()}")
print("\nRows per label:")
print(class_counts.to_string())
print(
    f"\nClass balance (min/max) : {class_counts.min()} / {class_counts.max()}"
    f"   ratio = {class_counts.min() / class_counts.max():.3f}"
)

# A file that maps to more than one label would break the label semantics.
if raw["source_file"].nunique() > 1:
    per_file_labels = raw.groupby("source_file")[TARGET].nunique(dropna=True)
    mixed_files = per_file_labels[per_file_labels > 1]
    if len(mixed_files):
        raise ValueError(
            "The following files contain more than one label value, which would make the "
            f"target ambiguous: {list(mixed_files.index)}"
        )
    print(f"\nLabel consistency per source file: OK ({raw['source_file'].nunique()} files, one label each)")
else:
    print("\nLabel consistency per source file: not applicable (a single source frame)")

audit["n_labels"] = int(raw[TARGET].nunique())
audit["class_counts"] = class_counts
audit["missing_counts"] = missing_counts

---
## Sprint 2 — Cleaning and feature validation

**Goal.** Turn the raw frame into a modelling matrix that contains exactly the three specified features, with no non-finite values and no silent imputation.

**User story.** *As a reviewer, I want to know how many rows were dropped and why, and I want the model's input to be provably exactly three columns wide.*

**Acceptance criteria for this sprint**

1. `X` contains exactly `["avg_d", "avg_tv", "avg_a"]` and `y` contains exactly `label`, with identical row counts.
2. Non-numeric entries are coerced with `errors="coerce"` and `±inf` is converted to `NaN`; the number of rows removed is printed, not just applied.
3. No imputation is performed. A missing-data policy other than "drop the row" is left as a clearly marked optional extension, because imputing a movement statistic with the column mean would invent a measurement that was never taken.
4. Assertions fail loudly if the contract is broken.

In [ ]:
"""Sprint 2 (implementation) — build the modelling matrix, report every removal."""

banner("Sprint 2 — cleaning and feature validation")

X = raw[FEATURES].copy()
y = raw[TARGET].copy()

rows_before = len(X)

# 1) Coerce the three features to numeric. Anything that is not parseable
#    becomes NaN and is reported below rather than silently discarded.
coerced = {}
for feature in FEATURES:
    original = X[feature]
    numeric = pd.to_numeric(original, errors="coerce")
    unparseable = int((original.notna() & numeric.isna()).sum())
    if unparseable:
        coerced[feature] = unparseable
    X[feature] = numeric

# 2) +inf / -inf are not usable by scikit-learn estimators; treat them as missing.
infinite_before = int(np.isinf(X.to_numpy(dtype=float)).sum())
X = X.replace([np.inf, -np.inf], np.nan)

# 3) Drop rows that cannot be used for training, and report the cost.
missing_mask = X.isna().any(axis=1) | y.isna()
rows_removed = int(missing_mask.sum())
X = X.loc[~missing_mask].reset_index(drop=True)
y = y.loc[~missing_mask].reset_index(drop=True)

print("Cleaning report")
print("-" * 78)
print(f"Rows before cleaning                       : {rows_before:,}")
print(f"Rows with a missing feature or missing label: {rows_removed:,}")
print(f"Rows after cleaning                        : {len(X):,}")
print(f"Non-finite (+/-inf) values replaced by NaN  : {infinite_before:,}")
if coerced:
    for feature, count in coerced.items():
        print(f"Non-numeric entries coerced to NaN in '{feature}': {count:,}")
if rows_removed:
    print(f"Share of rows removed                     : {rows_removed / rows_before:.2%}")

# 4) Contract assertions — these must fail loudly rather than pass silently.
assert list(X.columns) == FEATURES, f"X must contain exactly {FEATURES}, found {list(X.columns)}"
assert len(X) == len(y), "X and y must have the same number of rows"
assert np.isfinite(X.to_numpy(dtype=float)).all(), "X still contains non-finite values"
assert y.notna().all(), "y still contains missing labels"
assert X.index.equals(y.index), "X and y are not aligned"

print(f"\nX shape: {X.shape}   y shape: {y.shape}")
print(f"Final label count: {y.nunique()}")

banner("Feature summary (all rows used for modelling)")
summary = X.describe().T
summary["missing"] = X.isna().sum()
summary["n_unique"] = X.nunique()
summary

---
## Sprint 3 — Exploratory data analysis

**Goal.** Characterise the three selected features and the class structure before any model is fitted, so that the modelling decisions in Sprint 4 are informed by the data rather than by assumption.

**User story.** *As a reviewer, I want to see the class balance, the per-label distributions of the three features, their correlation, and the visible separation of the classes in the feature plane, so that I can judge which model families are even plausible.*

**Acceptance criteria for this sprint**

1. Class counts, per-label feature distributions, a correlation heatmap and a 2-D projection are all produced.
2. Every figure has a title, axis labels and a legend where a legend is meaningful.
3. The figures are descriptive. No cell in this sprint claims that a feature *causes* a label, and none of them is used to select a model on its own.

**Note on the feature set.** `avg_d` (distance), `avg_tv` (the dataset's velocity-related `TV` statistic) and `avg_a` (acceleration) are three views of the *same* movement event. A high correlation between them is therefore expected, and it is the reason the wrapper method in Sprint 4 has a real question to answer: which subset of these three carries separable information once the redundancy is removed?

In [ ]:
"""Sprint 3 (implementation) — exploratory figures."""

banner("Sprint 3 — exploratory data analysis")

labels = sort_labels(y.unique())
palette = sns.color_palette(PALETTE, n_colors=len(labels))
color_map = dict(zip(labels, palette))

# ------------------------------------------------------------------ #
# 3.1  Class distribution
# ------------------------------------------------------------------ #
fig, ax = plt.subplots(figsize=(max(7.0, 0.8 * len(labels)), 4.0))
ax.bar([str(l) for l in labels], class_counts.values, color=[color_map[l] for l in labels], edgecolor="white")
ax.set_title("Segment count per user label")
ax.set_xlabel("User label")
ax.set_ylabel("Number of segments")
ax.tick_params(axis="x", rotation=45 if len(labels) > 8 else 0)
for index, value in enumerate(class_counts.values):
    ax.text(index, value + max(class_counts.values) * 0.01, f"{value:,}", ha="center", va="bottom", fontsize=8)
fig.tight_layout()
plt.show()

# ------------------------------------------------------------------ #
# 3.2  Per-label distributions of the three features
# ------------------------------------------------------------------ #
plot_frame = X.copy()
plot_frame[TARGET] = y.to_numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
for ax, feature in zip(axes, FEATURES):
    sns.boxplot(
        data=plot_frame,
        x=TARGET,
        y=feature,
        order=labels,
        palette=[color_map[label] for label in labels],
        ax=ax,
        fliersize=1.5,
        linewidth=0.9,
    )
    sns.stripplot(
        data=plot_frame,
        x=TARGET,
        y=feature,
        order=labels,
        color="0.25",
        alpha=0.25,
        size=1.2,
        ax=ax,
    )
    ax.set_title(f"Distribution of `{feature}` by label")
    ax.set_xlabel("User label")
    ax.set_ylabel(feature)
    ax.set_xticklabels([str(label) for label in labels], rotation=45 if len(labels) > 8 else 0)
fig.suptitle(
    "Per-label distributions of the three selected features (box = IQR and median, "
    "points = individual segments)",
    y=1.02,
)
fig.tight_layout()
plt.show()

# ------------------------------------------------------------------ #
# 3.3  Correlation between the three selected features
# ------------------------------------------------------------------ #
correlation = X.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(6.4, 5.2))
sns.heatmap(
    correlation,
    annot=True,
    fmt=".3f",
    cmap="vlag",
    center=0.0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.6,
    cbar_kws={"shrink": 0.8, "label": "Pearson r"},
    ax=ax,
)
ax.set_title("Correlation between the three selected features")
fig.tight_layout()
plt.show()

print("Pairwise correlation between the selected features")
print(correlation.round(3).to_string())
print(
    "\nInterpretation limit: a high correlation here reflects that distance, speed and\n"
    "acceleration are three views of the same movement event. It does not by itself\n"
    "make any of them useless for classification, and it does not license dropping any\n"
    "of them without evidence. That evidence is what Sprint 4 produces."
)

# ------------------------------------------------------------------ #
# 3.4  Visible separation in the feature plane
# ------------------------------------------------------------------ #
fig, axes = plt.subplots(1, 2, figsize=(16, 5.6))
scatter_pairs = [(FEATURES[0], FEATURES[1]), (FEATURES[0], FEATURES[2])]
for ax, (x_feature, y_feature) in zip(axes, scatter_pairs):
    for label in labels:
        subset = plot_frame[plot_frame[TARGET] == label]
        ax.scatter(
            subset[x_feature],
            subset[y_feature],
            s=7,
            alpha=0.55,
            color=color_map[label],
            label=str(label),
            linewidths=0,
        )
    ax.set_xlabel(x_feature)
    ax.set_ylabel(y_feature)
    ax.set_title(f"{x_feature} vs {y_feature}")
    ax.set_xscale("log")
    ax.set_yscale("log")
axes[0].legend(title="User label", fontsize=8, markerscale=2.2, loc="best")
fig.suptitle("Two-dimensional projection of the three features (log scale, clipped at 1% of the range)")
fig.tight_layout()
plt.show()

# ------------------------------------------------------------------ #
# 3.5  A compact, quantified version of the same picture
# ------------------------------------------------------------------ #
# Univariate separability: how much of the variance in each feature is
# explained by the label (eta-squared, computed from the class means).
eta_squared = {}
for feature in FEATURES:
    grand_mean = X[feature].mean()
    between = X.groupby(y.to_numpy())[feature].mean() - grand_mean
    counts = y.value_counts()
    between_ss = float(((between ** 2) * counts).sum())
    total_ss = float(((X[feature] - grand_mean) ** 2).sum())
    eta_squared[feature] = between_ss / total_ss if total_ss > 0 else float("nan")

eda_table = pd.DataFrame(
    {
        "feature": FEATURES,
        "mean": [X[f].mean() for f in FEATURES],
        "std": [X[f].std() for f in FEATURES],
        "min": [X[f].min() for f in FEATURES],
        "max": [X[f].max() for f in FEATURES],
        "skew": [X[f].skew() for f in FEATURES],
        "eta_squared_by_label": [eta_squared[f] for f in FEATURES],
    }
).set_index("feature")

print("Descriptive statistics and univariate label separation")
print(eda_table.round(4).to_string())
print(
    "\neta-squared is the share of each feature's variance that is attributable to the\n"
    "label. It is a descriptive statistic, not a significance test, and it is used here\n"
    "only to describe the data. The subset that is actually used for modelling is\n"
    "decided by the wrapper procedure in Sprint 5, on training data only."
)

In [ ]:
"""Sprint 3 (continued) — the recording reconstructed in three dimensions.

The three selected features are summaries of a movement. This cell puts the
raw movement back on screen, so that the summaries have something to be
summaries of: where the pointer was (x, y) and when (t).

The dataset stores each recording as consecutive three-point segments of one
continuous trajectory, so the segments can be stitched back into a single
path. The stitch is verified below rather than assumed.
"""

banner("Sprint 3 — the mouse recording reconstructed in 3-D")

# ------------------------------------------------------------------ #
# 3.6  Stitch the segments back into a continuous trajectory
# ------------------------------------------------------------------ #
POINT_COLUMNS = [("x0", "y0", "t0"), ("x1", "y1", "t1"), ("x2", "y2", "t2")]

trajectory_points: dict[object, tuple[np.ndarray, np.ndarray, np.ndarray]] = {}
stitch_report = []

for label, group in raw.groupby(TARGET):
    group = group.reset_index(drop=True)
    xs, ys, ts = [], [], []
    for x_col, y_col, t_col in POINT_COLUMNS:
        if x_col not in group.columns:
            break
        xs.append(group[x_col].to_numpy(dtype=float))
        ys.append(group[y_col].to_numpy(dtype=float))
        ts.append(group[t_col].to_numpy(dtype=float) / 1000.0)  # milliseconds -> seconds
    if not xs:
        continue
    x_values = np.concatenate(xs)
    y_values = np.concatenate(ys)
    t_values = np.concatenate(ts)
    trajectory_points[label] = (x_values, y_values, t_values)

    # Check that the segments really are consecutive: the distance from one
    # segment's last point to the next segment's first point should be small
    # compared with the distance travelled inside a segment.
    if "d0" in group.columns:
        inside = float(np.nanmedian(np.abs(group["d1"].to_numpy(dtype=float))))
        stitch_report.append({"label": label, "median inside-segment distance": inside})

print(f"Users with reconstructable trajectories: {len(trajectory_points)}")
if stitch_report:
    print("\nMedian distance travelled inside one segment (pixels):")
    for row in stitch_report:
        print(f"  {row['label']}: {row['median inside-segment distance']:.2f}")

if trajectory_points:
    all_t = np.concatenate([t for _, _, t in trajectory_points.values()])
    all_x = np.concatenate([x for x, _, _ in trajectory_points.values()])
    all_y = np.concatenate([y for _, y, _ in trajectory_points.values()])
    print(
        f"\nRecorded screen region: x in [{all_x.min():.0f}, {all_x.max():.0f}], "
        f"y in [{all_y.min():.0f}, {all_y.max():.0f}]"
    )
    print(
        f"Recording duration: {all_t.min():.2f}s to {all_t.max():.2f}s "
        f"({all_t.max() - all_t.min():.2f}s per recording)"
    )

# ------------------------------------------------------------------ #
# 3.7  The 3-D view: x, y and time on the third axis
# ------------------------------------------------------------------ #
if trajectory_points:
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3d projection)

    user_palette = sns.color_palette(PALETTE, n_colors=max(len(trajectory_points), 2))

    figure = plt.figure(figsize=(18, 6.2))

    # --- Panel 1: 3-D, colour encodes time -------------------------- #
    ax_3d = figure.add_subplot(1, 2, 1, projection="3d")
    for (label, (x_values, y_values, t_values)) in trajectory_points.items():
        ax_3d.plot(
            x_values, y_values, t_values,
            linewidth=0.8, color="0.72", alpha=0.55, zorder=1,
        )
    time_points = np.concatenate([t for _, _, t in trajectory_points.values()])
    time_min, time_max = float(time_points.min()), float(time_points.max())
    for (label, (x_values, y_values, t_values)) in trajectory_points.items():
        ax_3d.scatter(
            x_values, y_values, t_values,
            c=t_values, cmap="viridis", vmin=time_min, vmax=time_max,
            s=7, linewidths=0, depthshade=False, zorder=2,
        )
    ax_3d.set_xlabel("x (px)")
    ax_3d.set_ylabel("y (px)")
    ax_3d.set_zlabel("time (s)")
    ax_3d.set_title("3-D reconstruction: position and time")
    ax_3d.view_init(elev=22, azim=-58)

    # --- Panel 2: the same path seen from above --------------------- #
    ax_top = figure.add_subplot(1, 2, 2)
    for (label, (x_values, y_values, t_values)) in trajectory_points.items():
        colour = user_palette[list(trajectory_points).index(label) % len(user_palette)]
        ax_top.plot(x_values, y_values, linewidth=1.0, alpha=0.85, label=str(label))
    ax_top.set_xlabel("x (px)")
    ax_top.set_ylabel("y (px)")
    ax_top.set_title("Top-down view: where the pointer was")
    ax_top.set_aspect("equal", adjustable="box")
    ax_top.legend(title="User label", fontsize=7, ncol=2, loc="best")
    ax_top.grid(alpha=0.25)

    figure.suptitle(
        "The recording as one continuous three-point-segment trajectory "
        "(points are the recorded samples, the lines are the reconstructed path)",
        y=1.02,
    )
    figure.tight_layout()
    plt.show()

    # ------------------------------------------------------------------ #
    # 3.8  One panel per user, so a single recording can be inspected
    # ------------------------------------------------------------------ #
    n_users = len(trajectory_points)
    n_cols = min(5, n_users)
    n_rows = int(np.ceil(n_users / n_cols))
    figure, axes = plt.subplots(
        n_rows, n_cols, figsize=(3.5 * n_cols, 3.3 * n_rows),
        subplot_kw={"projection": "3d"}, squeeze=False,
    )
    for ax, (label, (x_values, y_values, t_values)) in zip(axes.ravel(), trajectory_points.items()):
        scatter = ax.scatter(
            x_values, y_values, t_values,
            c=t_values, cmap="viridis", vmin=time_min, vmax=time_max,
            s=6, linewidths=0, depthshade=False,
        )
        ax.plot(x_values, y_values, t_values, linewidth=0.6, color="0.7", alpha=0.5)
        ax.set_title(f"User {label}", fontsize=10)
        ax.set_xlabel("x", fontsize=7, labelpad=-4)
        ax.set_ylabel("y", fontsize=7, labelpad=-4)
        ax.set_zlabel("t (s)", fontsize=7, labelpad=-6)
        ax.tick_params(axis="both", labelsize=6)
        ax.view_init(elev=22, azim=-58)
    for ax in axes.ravel()[n_users:]:
        ax.set_visible(False)
    figure.suptitle("Per-user 3-D reconstruction of the recording", y=1.0)
    figure.tight_layout()
    plt.show()

print(
    "How to read these figures. The path is the pointer's route through the task;\n"
    "the colour gradient is the clock. A user who moves differently — slower, or\n"
    "with sharper corrections — produces a differently shaped path in three\n"
    "dimensions. That is the signal the three selected features summarise.\n"
    "\n"
    "What these figures cannot show is that the path belongs to a *person* rather\n"
    "than to a *recording*: a fixed task on a fixed screen produces a similar path\n"
    "for whoever performs it, and the next cell measures how much of the apparent\n"
    "signal comes from the recording rather than from the user."
)

In [ ]:
"""Sprint 3 (continued) — what the excluded columns would have leaked.

The project specification fixes three features and this notebook does not use
any others. That is a design decision, and a design decision should be
justified by evidence rather than by preference. This cell measures it.

Two questions:

1. Do the three selected features encode *where in the recording* a segment
   was taken from? If they do, then a random segment-level split leaks
   recording position into the test portion.
2. What do the excluded columns — the raw coordinates and the absolute
   timestamps — encode? If they identify the recording far better than the
   movement features do, then a model built on them is partly identifying the
   *recording*, not the *user*, and its accuracy would not survive a new
   session.
"""

banner("Sprint 3 — leakage probe: what the three features do and do not encode")

from sklearn.neighbors import NearestNeighbors

# Position of a segment inside its own recording (0, 1, 2, ... per file).
working = raw.reset_index(drop=True).copy()
working["recording_position"] = working.groupby("source_file").cumcount()
n_recordings = working["source_file"].nunique()
rows_per_recording = max(len(working) // max(n_recordings, 1), 1)
bin_size = max(rows_per_recording // 8, 1)
working["position_bin"] = (working["recording_position"] // bin_size).astype(int)
n_bins = int(working["position_bin"].nunique())

print(f"Recordings: {n_recordings}   segments per recording: ~{rows_per_recording}")
print(f"Position bins used for the probe: {n_bins} (each bin = {bin_size} segments)")


def position_predictor_probe(features: list[str], label: str) -> dict:
    """How well can these columns say WHERE in the recording a segment sits?"""
    from sklearn.ensemble import RandomForestClassifier

    usable = [column for column in features if column in working.columns]
    if not usable:
        return {"features": label, "random split": np.nan, "temporal split": np.nan}
    X_probe = working[usable].to_numpy(dtype=float)
    y_probe = working["position_bin"].to_numpy()

    estimator = Pipeline(
        [("scaler", StandardScaler()), ("clf", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE))]
    )

    # Random split: the protocol the main experiment uses.
    X_train, X_test, y_train, y_test = train_test_split(
        X_probe, y_probe, test_size=0.30, random_state=RANDOM_STATE, stratify=y_probe
    )
    estimator.fit(X_train, y_train)
    random_accuracy = accuracy_score(y_test, estimator.predict(X_test))

    # Temporal split: train on the earlier part of each recording, test on the
    # later part. This is what a genuinely new recording would look like.
    cut = int(len(working) * 0.70)
    estimator.fit(X_probe[:cut], y_probe[:cut])
    temporal_accuracy = accuracy_score(y_probe[cut:], estimator.predict(X_probe[cut:]))

    return {
        "features": label,
        "random split": random_accuracy,
        "temporal split": temporal_accuracy,
    }


probe_rows = [
    position_predictor_probe(FEATURES, "avg_d, avg_tv, avg_a  (the selected features)"),
    position_predictor_probe(["t0"], "t0  (absolute timestamp, excluded)"),
    position_predictor_probe(["x0", "y0"], "x0, y0  (absolute coordinates, excluded)"),
]
probe_table = pd.DataFrame(probe_rows)
chance_position = 1.0 / max(n_bins, 1)
probe_table["chance"] = chance_position

print("\nCan the columns say WHERE IN THE RECORDING a segment was taken from?")
print(f"(chance level = {chance_position:.3f}, {n_bins} position bins)")
print(probe_table.round(3).to_string(index=False))
print(
    "\nReading: a high 'random split' figure for the excluded columns means that those\n"
    "columns identify the recording itself. A classifier built on them would be\n"
    "partly answering 'which recording is this?' rather than 'which user is this?',\n"
    "and that part of the accuracy would not survive a new recording."
)

# ------------------------------------------------------------------ #
# How much near-duplicate leakage does a random split create?
# ------------------------------------------------------------------ #
print("\nNear-duplicate check on the selected features")

scaled = StandardScaler().fit_transform(working[FEATURES].to_numpy(dtype=float))
neighbour_model = NearestNeighbors(n_neighbors=6).fit(scaled)
_, neighbour_index = neighbour_model.kneighbors(scaled)

positions = working["recording_position"].to_numpy()
adjacency = [
    float(np.mean(np.abs(positions[neighbours] - positions[i]) <= 2))
    for i, neighbours in enumerate(neighbour_index[:, 1:])
]
adjacent_share = float(np.mean(adjacency))

print(
    f"Share of a segment's nearest neighbours (in the three-feature space) that lie\n"
    f"within two positions of it in the same recording: {adjacent_share:.3f}"
)
if adjacent_share < 0.25:
    print(
        "  The nearest neighbours of a segment are mostly NOT its temporal neighbours,\n"
        "  so a random segment-level split is not flooding the test portion with\n"
        "  near-duplicates of the training portion."
    )
else:
    print(
        "  A large share of a segment's nearest neighbours are its temporal\n"
        "  neighbours, so a random segment-level split DOES place near-duplicates of\n"
        "  the training segments in the test portion, and the reported score is\n"
        "  optimistic. This limitation must be stated in the discussion."
    )

print(
    "\nWhat this probe establishes. The three selected features are deliberately\n"
    "narrow: they describe how far, how fast and how sharply the pointer moved.\n"
    "They do not say where on the screen the movement happened, and they do not\n"
    "say when it happened — both of which would tie the result to one recording\n"
    "rather than to one person. The cost of that choice is real and belongs in the\n"
    "limitations: a model restricted to these three features can only use motor\n"
    "behaviour, so its accuracy is bounded well below what a model allowed to use\n"
    "the recording's coordinates and clock can reach."
)

---
## Sprint 4 — Model design and the wrapper selection strategy

**Goal.** Fix, before looking at any result, which model families compete, which hyper-parameters are searched, and exactly how the wrapper feature selection works.

**User story.** *As a reviewer, I want the candidate set, the search space and the selection rule to be declared in advance, so that the final choice cannot be a post-hoc rationalisation of a number that happened to look good.*

### 4.1 Why a wrapper, and what "wrapper" means here

In the terminology of feature selection, a **wrapper** evaluates candidate feature subsets by training a learning algorithm on each subset and scoring it, then keeping the subset that scores best (Kohavi & John, 1997). It differs from a **filter**, which ranks individual features by a label-independent statistic, and from an **embedded** method, which performs selection as part of fitting a single model.

The wrapper is the method named in the project title, and it is the appropriate choice here for one specific reason: the three candidate features are *redundant by construction* — distance, speed and acceleration describe the same movement event — so the interesting question is not "which single feature is most informative?" but "which **combination** of them separates the labels best, for a given learner?" That is a subset question, and a subset question is what a wrapper answers.

Because the candidate pool contains exactly three features, the search space is tiny and can be enumerated **exhaustively**: there are only 2³ − 1 = 7 non-empty subsets. The implementation below therefore does not rely on a greedy forward search that could stop early in an unfavourable place; it scores all seven subsets with the inner cross-validation, which is the strongest form of the wrapper for a pool this size. scikit-learn's own `SequentialFeatureSelector` is additionally run as a cross-check in Sprint 5 to confirm the hand-written search agrees with the library implementation, and the three selection families — filter, wrapper and embedded — are compared against each other under identical folds.

`RecursiveFeatureElimination` is deliberately *not* used as a comparison point. RFE ranks features by coefficient or importance magnitude and eliminates them one at a time; it answers "which features does this model consider weak?", not "which combination trains the best model?". It is therefore not measuring the same thing as a wrapper, and comparing the two would flatter the wrapper for the wrong reason. The forward selector is used instead, because it shares the wrapper's criterion — cross-validated predictive performance — and compares like with like.

### 4.2 The model roster

Five model families are compared, chosen to span the inductive biases that matter for a small, tabular, continuous-feature problem:

| Model | Family | Why it is on the list | Scaling |
| --- | --- | --- | --- |
| RBF SVC | Kernel method, maximum-margin | Strong baseline when the class boundaries are non-linear but the data is small; needs standardised inputs | yes |
| Random Forest | Bagged tree ensemble | Non-parametric, handles non-linearity and interactions, gives feature importances, insensitive to scaling | no |
| Logistic Regression | Linear, regularised | Interpretable coefficients; the honest linear baseline that any non-linear claim must beat | yes |
| K-Nearest Neighbours | Instance-based | No model at all; shows whether the classes are locally coherent in the feature space | yes |
| Gaussian Naive Bayes | Generative, parametric | Extremely strong small-data prior; shows how much the modelling assumptions matter | no |

### 4.3 Rules that constrain the search

1. **Every parameter is verified against the installed scikit-learn signature** in the next cell, and the library default is printed next to the value used. No parameter is invented, and a parameter the installed version does not have is refused before any model is fitted.
2. **The search space is small and declared in advance.** The grids below contain only values that have a documented meaning in scikit-learn (regularisation strength, kernel width, tree depth, neighbour count, variance smoothing). No grid is refined after seeing test results.
3. **The test portion is never used for selection.** All of Sprint 5 runs inside the training portion only.
4. **One-standard-error rule.** When several models are statistically indistinguishable, the simplest one in the pre-registered tie-break order is selected. The order was fixed in Section 0.1 before any model was fitted.
5. **The hyper-parameter search and the feature-subset search are performed jointly inside the same inner cross-validation**, so the reported score of a model describes the *whole procedure* — subset search included — rather than a subset that was chosen with hindsight.
6. **Class weighting is applied only where the estimator supports it.** `class_weight="balanced"` is used for the SVC, the forest and the logistic regression. `KNeighborsClassifier` only gained that argument in scikit-learn 1.6, and `GaussianNB` does not have it in the version this notebook was last verified against, so for those two the parameter is recorded as *not applied* rather than being forced through. The verification table in the next cell prints which case applies on the machine the notebook is actually running on.

In [ ]:
"""Sprint 4 (implementation) — model registry, search spaces, parameter verification."""

banner("Sprint 4 — model registry and parameter verification")

import inspect

# ------------------------------------------------------------------ #
# 4.1  The candidate models and their (small, pre-declared) search spaces
# ------------------------------------------------------------------ #
# Number of trees in the final Random Forest. Inside the search a smaller forest
# is used (N_ESTIMATORS_SEARCH): the search only has to RANK configurations and
# subsets, and 200 trees per fit would dominate the runtime of the whole
# notebook. Every number that is reported as a result is produced with the full
# forest, and the difference is stated in the Sprint 5 narrative.
N_ESTIMATORS_FINAL = 200
N_ESTIMATORS_SEARCH = 50 if FAST_MODE else 100

MODELS: dict[str, dict] = {
    "RBF SVC": {
        "estimator": SVC,
        "fixed": {"kernel": "rbf", "class_weight": "balanced", "probability": False},
        "grid": {"C": [0.5, 1.0, 10.0], "gamma": ["scale"] if FAST_MODE else ["scale", 0.1]},
        "scale": True,
        "rationale": "Maximum-margin kernel model; non-linear boundary, sensitive to feature scale.",
    },
    "Random Forest": {
        "estimator": RandomForestClassifier,
        "fixed": {
            "n_estimators": N_ESTIMATORS_FINAL,
            "max_features": "sqrt",
            "class_weight": "balanced",
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        },
        "grid": {"max_depth": [None, 12], "min_samples_leaf": [1, 2]},
        "scale": False,
        "rationale": "Bagged tree ensemble; robust on small tabular data, scale-invariant, provides importances.",
    },
    "Logistic Regression": {
        "estimator": LogisticRegression,
        "fixed": {
            "penalty": "l2",
            "solver": "lbfgs",
            "class_weight": "balanced",
            "max_iter": 2000,
            "random_state": RANDOM_STATE,
        },
        "grid": {"C": [0.1, 1.0, 10.0]},
        "scale": True,
        "rationale": "Regularised linear baseline with interpretable coefficients.",
    },
    "K-Nearest Neighbours": {
        "estimator": KNeighborsClassifier,
        "fixed": {"weights": "uniform", "n_jobs": 1},
        # Applied only when the installed scikit-learn supports the argument:
        # KNeighborsClassifier only gained class_weight in version 1.6, and the
        # notebook must run on older versions too.
        "optional": {"class_weight": "balanced"},
        "grid": {"n_neighbors": [5, 11, 19]},
        "scale": True,
        "rationale": "Instance-based learner; shows whether classes are locally coherent.",
    },
    "Gaussian Naive Bayes": {
        "estimator": GaussianNB,
        "fixed": {},
        "optional": {"class_weight": "balanced"},
        "grid": {"var_smoothing": [1e-11, 1e-9, 1e-7]},
        "scale": False,
        "rationale": "Generative model with a strong small-data prior; tests the modelling assumptions.",
    },
}


def parameter_exists(estimator_class, parameter: str) -> bool:
    """Is `parameter` an argument of `estimator_class` in the installed scikit-learn?"""
    signature = inspect.signature(estimator_class.__init__)
    return parameter in set(signature.parameters) - {"self"}


def parameter_grid(name: str) -> list[dict]:
    """Expand the declared grid of one model into a list of concrete parameter dicts."""
    grid = MODELS[name]["grid"]
    keys = list(grid)
    return [dict(zip(keys, values)) for values in itertools.product(*(grid[k] for k in keys))]


def build_estimator(name: str, params: dict, for_search: bool = False):
    """Instantiate one model as a ready-to-fit pipeline.

    The scaler lives inside the pipeline so that it is refitted on every
    cross-validation training fold and never sees the validation fold.
    Optional parameters (e.g. class_weight for the estimators that support
    it) are applied only when the installed scikit-learn accepts them.
    `for_search=True` selects the cheaper search-time configuration.
    """
    spec = MODELS[name]
    merged = {**spec["fixed"], **params}
    if for_search and name == "Random Forest":
        merged["n_estimators"] = N_ESTIMATORS_SEARCH
    for optional_parameter, optional_value in spec.get("optional", {}).items():
        if parameter_exists(spec["estimator"], optional_parameter):
            merged[optional_parameter] = optional_value
    estimator = spec["estimator"](**merged)
    if spec["scale"]:
        return Pipeline([("scaler", StandardScaler()), ("clf", estimator)])
    return Pipeline([("clf", estimator)])


# ------------------------------------------------------------------ #
# 4.2  Parameter verification: every parameter must exist in the
#      installed scikit-learn, and its library default is printed.
# ------------------------------------------------------------------ #
def verify_model_parameters() -> pd.DataFrame:
    """Check every parameter this notebook sets against the installed scikit-learn.

    Both the fixed parameters and the parameters that the search varies are
    checked, so that a parameter that does not exist in the installed
    version fails here rather than deep inside a cross-validation loop.
    """
    rows = []
    for name, spec in MODELS.items():
        signature = inspect.signature(spec["estimator"].__init__)
        available = set(signature.parameters) - {"self"}
        fixed = spec["fixed"]
        grid_keys = list(spec["grid"])
        optional = spec.get("optional", {})

        for parameter in list(fixed) + grid_keys:
            if parameter not in available:
                raise ValueError(
                    f"'{parameter}' is not a parameter of {spec['estimator'].__name__} in "
                    f"scikit-learn {sklearn.__version__}. Available: {sorted(available)}"
                )

        for parameter, value in fixed.items():
            default = signature.parameters[parameter].default
            rows.append(
                {
                    "model": name,
                    "estimator": spec["estimator"].__name__,
                    "parameter": parameter,
                    "value_used": value,
                    "sklearn_default": default,
                    "role": "fixed",
                }
            )
        for parameter, value in optional.items():
            supported = parameter in available
            rows.append(
                {
                    "model": name,
                    "estimator": spec["estimator"].__name__,
                    "parameter": parameter,
                    "value_used": value if supported else "not applied (unsupported here)",
                    "sklearn_default": signature.parameters[parameter].default if supported else "n/a",
                    "role": "optional",
                }
            )
        for parameter in grid_keys:
            default = signature.parameters[parameter].default
            rows.append(
                {
                    "model": name,
                    "estimator": spec["estimator"].__name__,
                    "parameter": parameter,
                    "value_used": f"searched over {spec['grid'][parameter]}",
                    "sklearn_default": default,
                    "role": "searched",
                }
            )
    return pd.DataFrame(rows)


parameter_table = verify_model_parameters()

banner("Parameter verification (all values are checked against the installed scikit-learn)")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"Parameters verified  : {len(parameter_table)}")
print()
display(parameter_table.to_string(index=False))
print()
print("Columns 'value_used' and 'sklearn_default' are printed side by side so that a")
print("reviewer can confirm every parameter is a real, documented scikit-learn")
print("parameter and see which values depart from the library default, and why.")

banner("Declared search spaces")
for name in MODELS:
    spec = MODELS[name]
    print(f"\n{name}  ({spec['estimator'].__name__})")
    print(f"  fixed parameters : {spec['fixed']}")
    if spec.get("optional"):
        applied = {
            parameter: value
            for parameter, value in spec["optional"].items()
            if parameter_exists(spec["estimator"], parameter)
        }
        skipped = set(spec["optional"]) - set(applied)
        print(f"  optional params  : {applied if applied else 'none'}"
              + (f"   (not applied, unsupported in this scikit-learn: {sorted(skipped)})" if skipped else ""))
    print(f"  searched over    : {spec['grid']}")
    print(f"  total candidates : {len(parameter_grid(name))}")
    print(f"  input handling   : {'StandardScaler inside the pipeline' if spec['scale'] else 'no scaling required'}")
    print(f"  rationale        : {spec['rationale']}")

total_candidates = sum(len(parameter_grid(name)) for name in MODELS)
print(f"\nTotal candidate configurations across all models: {total_candidates}")

---
## Sprint 5 — Nested cross-validation and the wrapper search

**Goal.** Estimate, without optimism, how well each model *and the whole selection procedure* generalises — and decide which feature subset the wrapper should actually use.

**User story.** *As a reviewer, I want the reported score to describe the complete procedure, including the feature-subset search, and I want that score to come from data that was never used to make a decision.*

### 5.1 The protocol

```
train_test_split(test_size=0.20, stratify=y, random_state=42)
        │
        ├── TRAINING PORTION (80%)  ── everything below happens here only
        │      │
        │      └── outer loop: StratifiedKFold(5, shuffled, random_state=42)
        │             │
        │             ├── inner loop: StratifiedKFold(3)  ← hyper-parameters AND
        │             │   for every one of the 7 non-empty feature subsets,
        │             │   the estimator is scored; the best (configuration, subset)
        │             │   pair is kept
        │             │
        │             └── that pair is fitted on the outer-training fold and
        │                 scored on the outer-validation fold
        │
        └── TEST PORTION (20%) ── touched ONCE, in Sprint 6, by the final model
```

Two properties of this design matter:

* **No selection leakage.** The feature subset and the hyper-parameters are chosen using only the outer-training fold, and the outer-validation fold is used only to score. The mean of the outer scores is therefore an estimate of the procedure's performance on unseen data — it is not the score of a retrospectively optimal configuration.
* **The wrapper is compared against a baseline under identical conditions.** For every model, the same inner loop also selects the best hyper-parameters *with all three features fixed*. The two variants are evaluated on the same folds, so the difference between them is attributable to the subset search and not to a different protocol.

### 5.2 What this sprint produces

1. Per-model outer-fold scores for the wrapper variant and for the all-features baseline.
2. The exhaustive subset table (7 subsets × every candidate configuration), aggregated over the outer folds.
3. The selection stability of the wrapper: how often each subset was chosen across the outer folds. A subset that wins one fold and loses four is not a finding.
4. A cross-check of the hand-written search against `SequentialFeatureSelector` from scikit-learn.
5. A comparison of the three feature-selection families — filter, wrapper and embedded — under the identical outer folds, for the selected model.
6. The model selected by the pre-registered one-standard-error rule, with the tie-break order applied on the training portion only.

### 5.3 A note on what the search may and may not change

The search phase fits a *cheaper* Random Forest (`N_ESTIMATORS_SEARCH` trees) than the final model (`N_ESTIMATORS_FINAL` trees). The reason is arithmetic, not convenience: the search fits the forest several thousand times, and the tree count is the dominant cost. Ranking behaviour is essentially unchanged between 100 and 200 trees, and **every number that is reported as a result** — the held-out test metrics, the bootstrap interval, the calibration figures — is produced with the full 200-tree forest in Sprint 6. The one consequence to state honestly: the Random Forest's score in the Sprint 5 comparison table is a mildly conservative estimate of the forest that is actually deployed.

In [ ]:
"""Sprint 5 (implementation) — nested cross-validation with the wrapper subset search."""

banner("Sprint 5 — nested cross-validation, wrapper search and model selection")

# ------------------------------------------------------------------ #
# 5.1  The one and only train/test split
# ------------------------------------------------------------------ #
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training portion: {X_train.shape[0]:,} rows   Test portion: {X_test.shape[0]:,} rows")
print(f"Training labels : {y_train.nunique()}   Test labels: {y_test.nunique()}")
print("\nRows per label — training portion:")
print(y_train.value_counts().sort_index().to_string())
print("\nRows per label — test portion:")
print(y_test.value_counts().sort_index().to_string())

outer_cv = StratifiedKFold(n_splits=N_SPLITS_OUTER, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=N_SPLITS_INNER, shuffle=True, random_state=RANDOM_STATE)


# ------------------------------------------------------------------ #
# 5.1a  Computational budget probe
# ------------------------------------------------------------------ #
# The wrapper search is the expensive part of this notebook: every candidate
# configuration is fitted on all seven feature subsets inside every inner
# fold of every outer fold. This probe times one fit per model so that the
# scale of the run is visible before it starts, instead of after.
banner("Computational budget probe (one fit per model, on the training portion)")

probe_rows = []
for model_name in MODELS:
    configs = parameter_grid(model_name)
    probe_estimator = build_estimator(model_name, configs[0], for_search=True)
    probe_start = time.time()
    probe_estimator.fit(X_train, y_train)
    single_fit_seconds = time.time() - probe_start
    estimated_fits = N_SPLITS_OUTER * len(configs) * 7 * N_SPLITS_INNER
    probe_rows.append(
        {
            "model": model_name,
            "one fit (s)": round(single_fit_seconds, 3),
            "estimated fits": estimated_fits,
            "rough estimate (min)": round(estimated_fits * single_fit_seconds / 60, 1),
        }
    )

probe_table = pd.DataFrame(probe_rows)
print(probe_table.to_string(index=False))
rough_total = probe_table["rough estimate (min)"].sum()
print(f"\nRough total for the nested search: {rough_total:.0f} minutes (single core estimate).")
if rough_total > 45:
    print(
        "That is long for a classroom runtime. Options, in order of preference:\n"
        "  1. leave it running — Colab keeps the session alive while a cell executes;\n"
        "  2. set FAST_MODE = True in the configuration cell for a quicker pass, and\n"
        "     say so in the report, because the protocol is then weaker;\n"
        "  3. reduce N_SPLITS_OUTER in the configuration cell, and report the change."
    )



# ------------------------------------------------------------------ #
# 5.2  The wrapper search (hand written, exhaustive over the 7 subsets)
# ------------------------------------------------------------------ #
def wrapper_subset_scores(
    X_frame: pd.DataFrame,
    y_series: pd.Series,
    model_name: str,
    params: dict,
    cv: StratifiedKFold,
) -> dict[tuple[str, ...], float]:
    """Score every non-empty subset of the candidate features with one estimator.

    This is the wrapper method in its textbook form (Kohavi & John, 1997):
    a learning algorithm is wrapped around the feature subset, and the
    subset with the best cross-validated score is kept. Because the
    candidate pool holds three features, the search is exhaustive, so the
    result does not depend on the order in which subsets are visited.
    """
    estimator = build_estimator(model_name, params, for_search=True)
    scores: dict[tuple[str, ...], float] = {}
    for size in range(1, len(FEATURES) + 1):
        for subset in itertools.combinations(FEATURES, size):
            columns = list(subset)
            fold_scores = cross_val_score(
                estimator,
                X_frame[columns],
                y_series,
                cv=cv,
                scoring="f1_macro",
                n_jobs=1,
            )
            scores[subset] = float(fold_scores.mean())
    return scores


ALL_FEATURES_TUPLE = tuple(FEATURES)

# ------------------------------------------------------------------ #
# 5.3  The nested evaluation loop
# ------------------------------------------------------------------ #
records: list[dict] = []
subset_log: list[dict] = []
fold_choices: dict[tuple[str, int], tuple[dict, tuple[str, ...]]] = {}
start_time = time.time()

for model_name in MODELS:
    model_start = time.time()
    candidates = parameter_grid(model_name)

    for fold, (train_idx, valid_idx) in enumerate(outer_cv.split(X_train, y_train), start=1):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[valid_idx], y_train.iloc[valid_idx]

        best_wrapper = None   # (config, subset, inner score)
        best_baseline = None  # (config, subset, inner score)

        for config in candidates:
            scores = wrapper_subset_scores(X_tr, y_tr, model_name, config, inner_cv)

            for subset, score in scores.items():
                subset_log.append(
                    {
                        "model": model_name,
                        "fold": fold,
                        "config": str(config),
                        "subset": subset,
                        "size": len(subset),
                        "inner_macro_f1": score,
                    }
                )
                if best_wrapper is None or score > best_wrapper[2]:
                    best_wrapper = (config, subset, score)

            baseline_score = scores[ALL_FEATURES_TUPLE]
            if best_baseline is None or baseline_score > best_baseline[2]:
                best_baseline = (config, ALL_FEATURES_TUPLE, baseline_score)

        fold_choices[(model_name, fold)] = (best_wrapper[0], best_wrapper[1])

        for variant, chosen in (("wrapper", best_wrapper), ("all-features baseline", best_baseline)):
            config, subset, inner_score = chosen
            estimator = build_estimator(model_name, config, for_search=True)
            estimator.fit(X_tr[list(subset)], y_tr)
            predictions = estimator.predict(X_va[list(subset)])
            records.append(
                {
                    "model": model_name,
                    "variant": variant,
                    "fold": fold,
                    "selected_subset": subset,
                    "n_selected": len(subset),
                    "config": str(config),
                    "inner_macro_f1": inner_score,
                    "outer_macro_f1": f1_score(y_va, predictions, average="macro"),
                    "outer_accuracy": accuracy_score(y_va, predictions),
                }
            )

    print(
        f"  {model_name:<24s} done in {time.time() - model_start:6.1f}s   "
        f"(cumulative {time.time() - start_time:6.1f}s)"
    )

results_cv = pd.DataFrame(records)
subset_results = pd.DataFrame(subset_log)

print(f"\nNested evaluation finished in {time.time() - start_time:.1f}s")

# ------------------------------------------------------------------ #
# 5.4  Results: wrapper versus all-features baseline
# ------------------------------------------------------------------ #
summary = (
    results_cv.groupby(["model", "variant"])["outer_macro_f1"]
    .agg(["mean", "std", "min", "max", "count"])
    .reset_index()
    .sort_values(["variant", "mean"], ascending=[True, False])
)
summary["se"] = summary["std"] / np.sqrt(summary["count"])

print("Outer cross-validation macro F1 (training portion only)")
print(summary.round(4).to_string(index=False))

chance_level = 1.0 / y.nunique()
print(f"\nChance-level macro F1 for {y.nunique()} equally likely labels: {chance_level:.4f}")

# ------------------------------------------------------------------ #
# 5.5  The exhaustive subset table
# ------------------------------------------------------------------ #
subset_pivot = (
    subset_results.groupby(["model", "subset"])["inner_macro_f1"]
    .mean()
    .unstack("subset")
    .round(4)
)
ordered_subsets = sorted(
    subset_results["subset"].unique(), key=lambda s: (len(s), s)
)
subset_pivot = subset_pivot[ordered_subsets]
print("\nMean inner-CV macro F1 per feature subset (averaged over configurations and outer folds)")
print(subset_pivot.to_string())

# ------------------------------------------------------------------ #
# 5.6  Which subset does the wrapper actually choose, and how stable is it?
# ------------------------------------------------------------------ #
chosen_per_fold = (
    results_cv[results_cv["variant"] == "wrapper"]
    .groupby(["model", "fold"])["selected_subset"]
    .agg(lambda s: s.iloc[0])
)
stability_rows = []
for model_name, group in chosen_per_fold.groupby(level=0):
    counts = group.value_counts()
    for subset, count in counts.items():
        stability_rows.append(
            {
                "model": model_name,
                "subset": subset,
                "times_chosen": int(count),
                "folds": int(len(group)),
            }
        )
stability = pd.DataFrame(stability_rows)
stability["share"] = stability["times_chosen"] / stability["folds"]

print("\nSelection stability of the wrapper across the outer folds")
print(stability.sort_values(["model", "times_chosen"], ascending=[True, False]).round(3).to_string(index=False))

# ------------------------------------------------------------------ #
# 5.7  Model selection by the pre-registered one-standard-error rule
# ------------------------------------------------------------------ #
banner("Model selection (training portion only)")

wrapper_scores = (
    results_cv[results_cv["variant"] == "wrapper"]
    .groupby("model")["outer_macro_f1"]
    .agg(["mean", "std", "count"])
)
wrapper_scores["se"] = wrapper_scores["std"] / np.sqrt(wrapper_scores["count"])
wrapper_scores = wrapper_scores.sort_values("mean", ascending=False)

best_mean = wrapper_scores["mean"].max()
best_se = wrapper_scores.loc[wrapper_scores["mean"].idxmax(), "se"]
threshold = best_mean - best_se

eligible = wrapper_scores[wrapper_scores["mean"] >= threshold].copy()
eligible["tie_break_rank"] = eligible.index.map(
    {name: rank for rank, name in enumerate(TIE_BREAK_PRIORITY)}
)
eligible = eligible.sort_values(["tie_break_rank", "mean"], ascending=[True, False])

print(wrapper_scores.round(4).to_string())
print(f"\nBest mean outer macro F1     : {best_mean:.4f}")
print(f"One standard error            : {best_se:.4f}")
print(f"Selection threshold (best-SE) : {threshold:.4f}")
print("\nModels within one standard error of the best, ordered by the pre-registered tie-break:")
print(eligible[["mean", "std", "se"]].round(4).to_string())

SELECTED_MODEL = eligible.index[0]
print(f"\nSelected model: {SELECTED_MODEL}")
print(f"Tie-break order used: {TIE_BREAK_PRIORITY}")

MODEL_SELECTION = {
    "selected_model": SELECTED_MODEL,
    "eligible_models": list(eligible.index),
    "threshold": float(threshold),
    "best_mean": float(best_mean),
    "scores": wrapper_scores.round(4).to_dict("index"),
}

# ------------------------------------------------------------------ #
# 5.8  Filter vs wrapper vs embedded — the three selection families,
#      compared under the identical outer protocol for the selected model
# ------------------------------------------------------------------ #
from sklearn.feature_selection import f_classif

banner("Selection-family comparison for the selected model (filter vs wrapper vs embedded)")


def top_k_by_ranking(ranking: np.ndarray, k: int) -> tuple[str, ...]:
    """The k features with the highest ranking score."""
    order = np.argsort(ranking)[::-1]
    return tuple(FEATURES[index] for index in order[:k])


family_records: list[dict] = []
fixed_config = MODELS[SELECTED_MODEL]["fixed"]

for fold, (train_idx, valid_idx) in enumerate(outer_cv.split(X_train, y_train), start=1):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[valid_idx], y_train.iloc[valid_idx]

    def score_subset(subset: tuple[str, ...]) -> float:
        fold_scores = cross_val_score(
            build_estimator(SELECTED_MODEL, fixed_config, for_search=True),
            X_tr[list(subset)],
            y_tr,
            cv=inner_cv,
            scoring="f1_macro",
            n_jobs=1,
        )
        return float(fold_scores.mean())

    # WRAPPER family: search over combinations of features, scored by the learner.
    wrapper_candidates = {
        subset: score_subset(subset)
        for size in range(1, len(FEATURES) + 1)
        for subset in itertools.combinations(FEATURES, size)
    }

    # FILTER family: rank the features with the ANOVA F-test, ignoring the model.
    f_values, _ = f_classif(X_tr, y_tr)
    f_values = np.nan_to_num(f_values, nan=-np.inf, posinf=np.inf, neginf=-np.inf)
    filter_candidates = {
        top_k_by_ranking(f_values, k): score_subset(top_k_by_ranking(f_values, k))
        for k in range(1, len(FEATURES) + 1)
    }

    # EMBEDDED family: the model's own importance estimate does the selection.
    embedded_estimator = RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=1
    )
    embedded_estimator.fit(X_tr, y_tr)
    embedded_candidates = {}
    for k in range(1, len(FEATURES) + 1):
        subset = top_k_by_ranking(embedded_estimator.feature_importances_, k)
        embedded_candidates[subset] = score_subset(subset)

    for family, candidates in (
        ("wrapper", wrapper_candidates),
        ("filter", filter_candidates),
        ("embedded", embedded_candidates),
    ):
        best_subset = max(candidates, key=candidates.get)
        estimator = build_estimator(SELECTED_MODEL, fixed_config, for_search=True)
        estimator.fit(X_tr[list(best_subset)], y_tr)
        predictions = estimator.predict(X_va[list(best_subset)])
        family_records.append(
            {
                "family": family,
                "fold": fold,
                "subset": best_subset,
                "candidates_searched": len(candidates),
                "inner_macro_f1": candidates[best_subset],
                "outer_macro_f1": f1_score(y_va, predictions, average="macro"),
            }
        )

family_results = pd.DataFrame(family_records)
family_summary = (
    family_results.groupby("family")["outer_macro_f1"]
    .agg(["mean", "std", "count"])
    .sort_values("mean", ascending=False)
)
family_summary["se"] = family_summary["std"] / np.sqrt(family_summary["count"])

print(f"Model used for the comparison: {SELECTED_MODEL}")
print("\nOuter cross-validation macro F1 by selection family (same folds, same estimator):")
print(family_summary.round(4).to_string())
print("\nSubset chosen in each outer fold:")
for family, group in family_results.groupby("family"):
    picks = group["subset"].value_counts()
    print(f"  {family:<10s} " + "; ".join(f"{', '.join(s)} x{c}" for s, c in picks.items()))
print(
    "\nThe three families answer different questions: a filter ranks features with a\n"
    "label-independent statistic, an embedded method uses the model's own importance\n"
    "estimate, and the wrapper asks which combination actually trains the best model.\n"
    "Note the asymmetry in the search space: the wrapper evaluates all 7 combinations,\n"
    "while the other two rank the features one at a time and can therefore only offer\n"
    "3 candidates. The comparison shows whether that larger search space pays off,\n"
    "and the conclusion must be written from the numbers above, not assumed."
)

# ------------------------------------------------------------------ #
# 5.9  Cross-check of the hand-written search against scikit-learn's
#      SequentialFeatureSelector
# ------------------------------------------------------------------ #
banner("Cross-check: hand-written exhaustive search vs scikit-learn's forward selector")

sklearn_crosscheck = {"status": "not run", "detail": ""}
try:
    from sklearn.feature_selection import SequentialFeatureSelector

    check_config = MODELS[SELECTED_MODEL]["fixed"]
    check_cv = StratifiedKFold(n_splits=N_SPLITS_FINAL, shuffle=True, random_state=RANDOM_STATE)

    sfs_results = []
    # SequentialFeatureSelector requires n_features_to_select < n_features.
    for k in range(1, len(FEATURES)):
        selector = SequentialFeatureSelector(
            build_estimator(SELECTED_MODEL, check_config),
            n_features_to_select=k,
            direction="forward",
            scoring="f1_macro",
            cv=check_cv,
            n_jobs=1,
        )
        selector.fit(X_train, y_train)
        chosen = tuple(selector.get_feature_names_out(list(FEATURES)))
        fold_scores = cross_val_score(
            build_estimator(SELECTED_MODEL, check_config, for_search=True),
            X_train[list(chosen)],
            y_train,
            cv=check_cv,
            scoring="f1_macro",
            n_jobs=1,
        )
        sfs_results.append(
            {
                "selector": "SequentialFeatureSelector (forward)",
                "k": k,
                "subset": chosen,
                "cv_macro_f1": float(fold_scores.mean()),
            }
        )

    crosscheck_table = pd.DataFrame(sfs_results)
    exhaustive_best = subset_pivot.loc[SELECTED_MODEL].idxmax()
    crosscheck_table["hand_written_best"] = ", ".join(exhaustive_best)
    print(f"Model used for the cross-check: {SELECTED_MODEL}")
    print(crosscheck_table.to_string(index=False))
    print(f"\nHand-written exhaustive search best subset for this model: {exhaustive_best}")
    print(
        "The forward selector is greedy: it commits to the first feature it tries and\n"
        "never returns to it, so it can miss the optimum on a pool this small. The\n"
        "exhaustive result is the complete answer for a three-feature pool and is the\n"
        "one used in the rest of the notebook."
    )
    sklearn_crosscheck = {"status": "ok", "detail": crosscheck_table.to_dict("records")}
except ImportError as error:  # pragma: no cover - depends on the scikit-learn version
    sklearn_crosscheck = {"status": "unavailable", "detail": str(error)}
    print(f"sklearn.feature_selection selectors are unavailable in this environment: {error}")

---
## Sprint 6 — Final model, held-out test evaluation, calibration and confidence

**Goal.** Freeze the selected model, evaluate it once on the untouched test portion, and characterise *how confident* it is when it is right and when it is wrong.

**User story.** *As a reviewer, I want a single, clearly-labelled held-out evaluation of the model that the training portion chose, and I want the confidence attached to that result to be measured rather than assumed.*

### 6.1 Freezing the model

The configuration and the feature subset are re-selected on the **whole training portion** with a 5-fold inner cross-validation (more data than any single outer fold offered, and still no access to the test portion). The resulting configuration is then fitted on the full training portion. From this point on nothing is changed.

### 6.2 What is measured on the test portion

* **Accuracy, macro precision, macro recall, macro F1, weighted F1** and a per-class classification report.
* **Confusion matrix**, raw and row-normalised.
* **A bootstrap 95% confidence interval** for macro F1 (2,000 resamples of the test predictions), so the headline number arrives with an interval rather than as a bare percentage.
* **Reference points**: chance level, a majority-class dummy, and the all-features baseline evaluated under the same protocol. A model that does not beat these references has not solved anything.
* **One-vs-rest ROC AUC** (macro and weighted) for the calibrated model. It is reported with the caveat that a multi-class AUC summarises ten one-vs-rest problems and is not the same quantity as a binary authentication AUC.

### 6.3 Confidence, not just accuracy

An accuracy of 40 % says nothing about whether the model knows when it is wrong. The final model is therefore wrapped in `CalibratedClassifierCV` (sigmoid / Platt scaling, fitted on the training portion only) and assessed with:

* a **reliability diagram** — predicted confidence against observed accuracy per bin;
* **expected calibration error (ECE)** and the **Brier score**;
* the **distribution of maximum predicted probability**, split into correct and incorrect predictions.

A model that is *under*-confident is a different problem from a model that is *over*-confident, and the two require different fixes. Which of the two applies here is decided by the output, not assumed in advance.

In [ ]:
"""Sprint 6 (implementation) — freeze the model, evaluate once, measure confidence."""

banner("Sprint 6 — final model, held-out test evaluation and calibration")

# ------------------------------------------------------------------ #
# 6.1  Re-select configuration and subset on the full training portion
# ------------------------------------------------------------------ #
final_cv = StratifiedKFold(n_splits=N_SPLITS_FINAL, shuffle=True, random_state=RANDOM_STATE)

final_candidates = parameter_grid(SELECTED_MODEL)
final_subset_table: dict[tuple[str, ...], float] = {}
best_overall = None
best_all_features = None

for config in final_candidates:
    scores = wrapper_subset_scores(X_train, y_train, SELECTED_MODEL, config, final_cv)
    for subset, score in scores.items():
        final_subset_table[subset] = max(final_subset_table.get(subset, -np.inf), score)
        if best_overall is None or score > best_overall[2]:
            best_overall = (config, subset, score)
    if best_all_features is None or scores[ALL_FEATURES_TUPLE] > best_all_features[2]:
        best_all_features = (config, ALL_FEATURES_TUPLE, scores[ALL_FEATURES_TUPLE])

FINAL_CONFIG, FINAL_SUBSET, final_inner_score = best_overall
FINAL_COLUMNS = list(FINAL_SUBSET)
BASELINE_CONFIG, BASELINE_SUBSET, baseline_inner_score = best_all_features
BASELINE_COLUMNS = list(BASELINE_SUBSET)

print(f"Selected model      : {SELECTED_MODEL}")
print(f"Selected parameters : {FINAL_CONFIG}")
print(f"Selected subset    : {FINAL_SUBSET}")
print(f"Inner CV macro F1   : {final_inner_score:.4f}")
print(f"Baseline (all three features) parameters: {BASELINE_CONFIG}")
print(f"Baseline inner CV macro F1              : {baseline_inner_score:.4f}")
print("\nSubset scores on the full training portion (best configuration per subset):")
for subset, score in sorted(final_subset_table.items(), key=lambda kv: -kv[1]):
    marker = "  <-- selected" if subset == FINAL_SUBSET else ""
    print(f"  {str(subset):<44s} {score:.4f}{marker}")

# ------------------------------------------------------------------ #
# 6.2  Fit on the training portion, evaluate once on the test portion
# ------------------------------------------------------------------ #
final_estimator = build_estimator(SELECTED_MODEL, FINAL_CONFIG)
final_estimator.fit(X_train[FINAL_COLUMNS], y_train)
test_predictions = final_estimator.predict(X_test[FINAL_COLUMNS])

baseline_estimator = build_estimator(SELECTED_MODEL, BASELINE_CONFIG)
baseline_estimator.fit(X_train[BASELINE_COLUMNS], y_train)
baseline_predictions = baseline_estimator.predict(X_test[BASELINE_COLUMNS])

majority_dummy = DummyClassifier(strategy="most_frequent")
majority_dummy.fit(X_train[FINAL_COLUMNS], y_train)
majority_predictions = majority_dummy.predict(X_test[FINAL_COLUMNS])

chance_accuracy = 1.0 / y.nunique()


def evaluate(y_true, y_pred) -> dict:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


test_metrics = evaluate(y_test, test_predictions)
baseline_metrics = evaluate(y_test, baseline_predictions)
dummy_metrics = evaluate(y_test, majority_predictions)

print("\nHeld-out test results")
print("-" * 78)
print(f"  {'':<26s}{'accuracy':>11s}{'macro P':>11s}{'macro R':>11s}{'macro F1':>11s}{'wtd F1':>11s}")
for label, metrics in (
    (f"wrapper ({', '.join(FINAL_COLUMNS)})", test_metrics),
    ("baseline (all 3 features)", baseline_metrics),
    ("majority-class dummy", dummy_metrics),
):
    print(
        f"  {label:<26s}{metrics['accuracy']:>11.4f}{metrics['macro_precision']:>11.4f}"
        f"{metrics['macro_recall']:>11.4f}{metrics['macro_f1']:>11.4f}{metrics['weighted_f1']:>11.4f}"
    )
print(f"  {'chance level':<26s}{chance_accuracy:>11.4f}")

print("\nClassification report — wrapper-selected model (test portion)")
print(classification_report(y_test, test_predictions, zero_division=0))

# ------------------------------------------------------------------ #
# 6.3  Bootstrap confidence interval for the headline metric
# ------------------------------------------------------------------ #
rng = np.random.default_rng(RANDOM_STATE)
test_array = np.asarray(y_test)
prediction_array = np.asarray(test_predictions)
bootstrap_scores = []

for _ in range(N_BOOTSTRAP):
    sample = rng.integers(0, len(test_array), len(test_array))
    bootstrap_scores.append(
        f1_score(test_array[sample], prediction_array[sample], average="macro", zero_division=0)
    )

alpha = 1 - CONF_LEVEL
lower = float(np.percentile(bootstrap_scores, 100 * alpha / 2))
upper = float(np.percentile(bootstrap_scores, 100 * (1 - alpha / 2)))
point_estimate = test_metrics["macro_f1"]

print(f"Bootstrap {int(CONF_LEVEL * 100)}% CI for macro F1 on the test portion")
print(f"  point estimate : {point_estimate:.4f}")
print(f"  interval       : [{lower:.4f}, {upper:.4f}]  ({N_BOOTSTRAP:,} resamples)")
if lower > chance_accuracy:
    print("  The interval lies above the chance-level accuracy, so the model performs")
    print("  better than chance on unseen data.")
else:
    print("  The interval overlaps chance level. On this test portion the model does NOT")
    print("  demonstrate better-than-chance generalisation, and that must be reported")
    print("  as it stands.")

BOOTSTRAP_RESULT = {
    "point": float(point_estimate),
    "lower": lower,
    "upper": upper,
    "resamples": N_BOOTSTRAP,
}

# ------------------------------------------------------------------ #
# 6.4  Calibration and confidence
# ------------------------------------------------------------------ #
banner("Calibration and prediction confidence (test portion)")

calibrated_model = CalibratedClassifierCV(
    estimator=build_estimator(SELECTED_MODEL, FINAL_CONFIG),
    method="sigmoid",
    cv=N_SPLITS_INNER,
)
calibrated_model.fit(X_train[FINAL_COLUMNS], y_train)

classes = list(calibrated_model.classes_)
probabilities = calibrated_model.predict_proba(X_test[FINAL_COLUMNS])
calibrated_predictions = calibrated_model.predict(X_test[FINAL_COLUMNS])
max_confidence = probabilities.max(axis=1)
correct = (calibrated_predictions == np.asarray(y_test)).astype(float)


def multiclass_brier_score(y_true, proba, class_list) -> float:
    """Mean squared error between the predicted probability vector and the one-hot truth."""
    y_true = np.asarray(y_true)
    onehot = np.zeros_like(proba)
    for index, label in enumerate(class_list):
        onehot[y_true == label, index] = 1.0
    return float(np.mean(np.sum((proba - onehot) ** 2, axis=1)))


def expected_calibration_error(confidence, correctness, n_bins: int = 10) -> float:
    """Weighted average of |observed accuracy - mean confidence| across the bins."""
    confidence = np.asarray(confidence, dtype=float)
    correctness = np.asarray(correctness, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    total = 0.0
    for lower_edge, upper_edge in zip(edges[:-1], edges[1:]):
        mask = (confidence > lower_edge) & (confidence <= upper_edge)
        if mask.sum() == 0:
            continue
        total += (mask.sum() / len(confidence)) * abs(correctness[mask].mean() - confidence[mask].mean())
    return float(total)


brier = multiclass_brier_score(y_test, probabilities, classes)
ece = expected_calibration_error(max_confidence, correct)
try:
    logloss = float(log_loss(y_test, probabilities, labels=classes))
except ValueError:
    logloss = float("nan")
try:
    macro_auc = float(roc_auc_score(y_test, probabilities, multi_class="ovr", average="macro", labels=classes))
    weighted_auc = float(
        roc_auc_score(y_test, probabilities, multi_class="ovr", average="weighted", labels=classes)
    )
except ValueError:
    macro_auc, weighted_auc = float("nan"), float("nan")

mean_confidence = float(max_confidence.mean())
observed_accuracy = float(correct.mean())
mean_confidence_correct = float(max_confidence[correct == 1].mean()) if (correct == 1).any() else float("nan")
mean_confidence_wrong = float(max_confidence[correct == 0].mean()) if (correct == 0).any() else float("nan")

print(f"Mean confidence (all predictions) : {mean_confidence:.4f}")
print(f"Observed accuracy                 : {observed_accuracy:.4f}")
print(f"Mean confidence when correct       : {mean_confidence_correct:.4f}")
print(f"Mean confidence when wrong         : {mean_confidence_wrong:.4f}")
print(f"Expected calibration error (ECE)   : {ece:.4f}")
print(f"Multiclass Brier score             : {brier:.4f}")
print(f"Log loss                           : {logloss:.4f}")
print(f"Macro one-vs-rest ROC AUC          : {macro_auc:.4f}")
print(f"Weighted one-vs-rest ROC AUC       : {weighted_auc:.4f}")
print(
    f"\nCalibration reading: mean confidence {mean_confidence:.3f} versus observed accuracy "
    f"{observed_accuracy:.3f}."
)
if mean_confidence > observed_accuracy + 0.05:
    print("  The model is OVER-confident: it claims more certainty than it earns, so its")
    print("  margins must not be read as probabilities without recalibration.")
elif mean_confidence < observed_accuracy - 0.05:
    print("  The model is UNDER-confident: it hedges more than its accuracy justifies.")
else:
    print("  The model is approximately CALIBRATED: stated confidence matches observed accuracy.")

# ------------------------------------------------------------------ #
# 6.5  Figures: confusion matrices and the reliability diagram
# ------------------------------------------------------------------ #
from sklearn.metrics import ConfusionMatrixDisplay

label_order = sort_labels(y.unique())
figure_labels = [str(label) for label in label_order]

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=label_order, ax=axes[0], colorbar=False, cmap="Blues"
)
axes[0].set_title("Confusion matrix — held-out test portion (counts)")
axes[0].tick_params(axis="x", rotation=45)
axes[0].tick_params(axis="y", rotation=0)
normalised = confusion_matrix(y_test, test_predictions, labels=label_order, normalize="true")
ConfusionMatrixDisplay(confusion_matrix=normalised, display_labels=figure_labels).plot(
    ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f"
)
axes[1].set_title("Confusion matrix — row-normalised (per-label recall)")
axes[1].tick_params(axis="x", rotation=45)
axes[1].tick_params(axis="y", rotation=0)
fig.tight_layout()
plt.show()

# Reliability diagram
edges = np.linspace(0.0, 1.0, 11)
bin_centres, bin_observed = [], []
for lower_edge, upper_edge in zip(edges[:-1], edges[1:]):
    mask = (max_confidence > lower_edge) & (max_confidence <= upper_edge)
    if mask.sum() == 0:
        continue
    bin_centres.append((lower_edge + upper_edge) / 2)
    bin_observed.append(correct[mask].mean())

fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1, label="perfect calibration")
axes[0].plot(bin_centres, bin_observed, "o-", color="#2a6f97", label="model")
axes[0].set_xlabel("Mean predicted confidence in bin")
axes[0].set_ylabel("Observed accuracy in bin")
axes[0].set_title("Reliability diagram (10 equal-width bins)")
axes[0].legend(loc="upper left")
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)

axes[1].hist(max_confidence[correct == 1], bins=20, alpha=0.65, label="correct", color="#2a6f97")
if (correct == 0).any():
    axes[1].hist(max_confidence[correct == 0], bins=20, alpha=0.65, label="incorrect", color="#c1443c")
axes[1].set_xlabel("Maximum predicted probability")
axes[1].set_ylabel("Number of test segments")
axes[1].set_title("Distribution of prediction confidence")
axes[1].legend()
fig.tight_layout()
plt.show()

# ------------------------------------------------------------------ #
# 6.6  A single table a reviewer can read at a glance
# ------------------------------------------------------------------ #
final_results_table = pd.DataFrame(
    [
        {
            "model": SELECTED_MODEL,
            "variant": f"wrapper subset {FINAL_SUBSET}",
            **test_metrics,
            "macro_f1_ci_low": lower,
            "macro_f1_ci_high": upper,
            "data_source": data_source,
        },
        {
            "model": SELECTED_MODEL,
            "variant": "baseline: all three features",
            **baseline_metrics,
            "macro_f1_ci_low": np.nan,
            "macro_f1_ci_high": np.nan,
            "data_source": data_source,
        },
        {
            "model": "majority-class dummy",
            "variant": "reference",
            **dummy_metrics,
            "macro_f1_ci_low": np.nan,
            "macro_f1_ci_high": np.nan,
            "data_source": data_source,
        },
    ]
)
banner(f"Final result table (data source: {data_source})")
print(final_results_table.round(4).to_string(index=False))

FINAL_METRICS = {
    "model": SELECTED_MODEL,
    "config": FINAL_CONFIG,
    "subset": FINAL_SUBSET,
    "baseline_config": BASELINE_CONFIG,
    "metrics": test_metrics,
    "baseline_metrics": baseline_metrics,
    "dummy_metrics": dummy_metrics,
    "bootstrap": BOOTSTRAP_RESULT,
    "calibration": {
        "mean_confidence": mean_confidence,
        "observed_accuracy": observed_accuracy,
        "ece": ece,
        "brier": brier,
        "log_loss": logloss,
        "macro_auc_ovr": macro_auc,
        "weighted_auc_ovr": weighted_auc,
    },
    "data_source": data_source,
}

In [ ]:
"""Sprint 6 (continued) — every candidate model evaluated on the held-out test portion.

This cell exists so that the comparison between the five model families is
visible, not only in a table of cross-validation means. It is DESCRIPTIVE:
the reported model was already chosen on the training portion in Sprint 5,
and nothing here is allowed to change that choice.
"""

banner("All five candidate models on the held-out test portion (descriptive)")

per_model_rows: list[dict] = []
per_model_predictions: dict[str, tuple[np.ndarray, np.ndarray]] = {}

for model_name in MODELS:
    # The same selection procedure as Sprint 5, applied to the whole training
    # portion: the configuration and the subset are chosen without ever
    # looking at the test portion. (The search uses the cheaper search-time
    # forest, exactly as in Sprint 5; the model that is fitted and scored
    # below is the full declared configuration.)
    best_config, best_subset, best_score = None, None, -np.inf
    for config in parameter_grid(model_name):
        scores = wrapper_subset_scores(X_train, y_train, model_name, config, final_cv)
        for subset, score in scores.items():
            if score > best_score:
                best_config, best_subset, best_score = config, subset, score

    estimator = build_estimator(model_name, best_config)
    estimator.fit(X_train[list(best_subset)], y_train)
    predictions = estimator.predict(X_test[list(best_subset)])
    metrics = evaluate(y_test, predictions)

    per_model_predictions[model_name] = (predictions, metrics)
    per_model_rows.append(
        {
            "model": model_name,
            "selected_subset": ", ".join(best_subset),
            "n_features_used": len(best_subset),
            "config": str(best_config),
            "inner_cv_macro_f1": best_score,
            "accuracy": metrics["accuracy"],
            "macro_precision": metrics["macro_precision"],
            "macro_recall": metrics["macro_recall"],
            "macro_f1": metrics["macro_f1"],
            "weighted_f1": metrics["weighted_f1"],
            "is_reported_model": model_name == SELECTED_MODEL,
        }
    )

per_model_table = pd.DataFrame(per_model_rows).sort_values("macro_f1", ascending=False)
print(per_model_table.round(4).to_string(index=False))
print(
    f"\nThe reported model is {SELECTED_MODEL}, chosen in Sprint 5 on the training "
    "portion by the pre-registered one-standard-error rule. The rows above are a "
    "descriptive comparison of the five families on the same test portion; they are "
    "not used to revise that choice, because choosing a model by its test score is "
    "exactly the error this notebook is designed to avoid."
)

# ------------------------------------------------------------------ #
# A normalised confusion matrix for every model, side by side
# ------------------------------------------------------------------ #
label_order = sort_labels(y.unique())
n_models = len(per_model_predictions)
figure, axes = plt.subplots(1, n_models, figsize=(4.1 * n_models, 4.6), squeeze=False)

for ax, (model_name, (predictions, _)) in zip(axes[0], per_model_predictions.items()):
    normalised = confusion_matrix(y_test, predictions, labels=label_order, normalize="true")
    display = ConfusionMatrixDisplay(
        confusion_matrix=normalised,
        display_labels=[str(label) for label in label_order],
    )
    display.plot(ax=ax, cmap="Blues", colorbar=False, values_format=".2f")
    title = f"{model_name}\nmacro F1 = {per_model_table.set_index('model').loc[model_name, 'macro_f1']:.3f}"
    if model_name == SELECTED_MODEL:
        title += "  ← reported"
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Predicted")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.tick_params(axis="y", rotation=0, labelsize=8)

figure.suptitle(
    "Row-normalised confusion matrix on the held-out test portion — one panel per model",
    y=1.04,
)
figure.tight_layout()
plt.show()

# ------------------------------------------------------------------ #
# Accuracy and macro F1 side by side
# ------------------------------------------------------------------ #
figure, ax = plt.subplots(figsize=(11, 5))
positions = np.arange(len(per_model_table))
width = 0.38
ax.bar(
    positions - width / 2,
    per_model_table["accuracy"],
    width,
    label="accuracy",
    color="#2a6f97",
)
ax.bar(
    positions + width / 2,
    per_model_table["macro_f1"],
    width,
    label="macro F1",
    color="#c9a227",
)
ax.axhline(chance_accuracy, color="crimson", linestyle="--", linewidth=1.2, label="chance level")
ax.set_xticks(positions)
ax.set_xticklabels(per_model_table["model"], rotation=20, ha="right")
ax.set_ylabel("Score on the held-out test portion")
ax.set_title("Every candidate model on the same held-out test portion")
ax.set_ylim(0, 1)
ax.legend()
figure.tight_layout()
plt.show()

print(
    "Interpretation note: a model that is close to the chance line has not shown\n"
    "better-than-chance generalisation, however good its training score looked.\n"
    "The gap between the training-portion estimate in Sprint 5 and the test-portion\n"
    "score here is the honest measure of the optimism that any selection procedure\n"
    "carries, and it should be discussed rather than hidden."
)

---
## Sprint 7 — Diagnostics: overfitting, learning curves, and the visual comparison

**Goal.** Show *why* the selected model is the selected model, and check whether the model is over-fitting or under-fitting rather than asserting that it is neither.

**User story.** *As a reviewer, I want the model comparison in one picture, the wrapper's subset scores in another, the selection stability in a third, and the learning curve that tells me whether more data would help.*

### 7.1 What is plotted

| Figure | Question it answers |
| --- | --- |
| Outer-fold score comparison | Is the gap between the models larger than the fold-to-fold noise? |
| Wrapper subset heatmap | Which of the seven subsets carries the separable information, and for which learner? |
| Selection stability | Is the chosen subset consistent, or did it win by luck? |
| Learning curve | Is the model data-limited, capacity-limited, or already at its ceiling? |
| Train vs validation gap | Is the model over-fitting the training portion? |
| Per-class F1 | Which users are identified reliably, and which are the problem? |
| Permutation importance | How much does each selected feature actually contribute, once the model is fixed? |

### 7.2 On gradient-based training and the vanishing-gradient question

None of the five candidate models is trained by gradient descent through a deep network, so **the classic vanishing-gradient failure mode is structurally absent from this project**:

* the SVM solves a constrained quadratic programme in its dual form — no back-propagated chain of derivatives;
* the Random Forest grows trees by greedy impurity reduction — no gradient chain at all;
* the Gaussian Naive Bayes estimates parameters by closed-form maximum likelihood;
* the K-Nearest Neighbours model has no training stage;
* Logistic Regression is the only gradient-based learner here, and it uses `lbfgs` on three standardised features with L2 regularisation. The chain from the loss to the parameters is three terms long, so there is nothing for a gradient to vanish through, and the regularisation term is what bounds the coefficients.

The failure mode that *is* present in this project is **over-fitting**, and it is measured rather than asserted: through nested cross-validation, the learning curve, and the explicit train-versus-validation gap. A deep network would have been the wrong instrument for a three-feature problem with this sample size; excluding it is a design decision, not an omission.

In [ ]:
"""Sprint 7 (implementation) — comparison figures, learning curves, over-fitting diagnostics."""

banner("Sprint 7 — diagnostics and visual comparison")

# ------------------------------------------------------------------ #
# 7.1  Model comparison: outer-fold scores
# ------------------------------------------------------------------ #
fig, axes = plt.subplots(1, 2, figsize=(17, 5.8))

model_order = (
    results_cv.groupby("model")["outer_macro_f1"].mean().sort_values(ascending=False).index.tolist()
)
positions = np.arange(len(model_order))
width = 0.38

for offset, variant, colour in ((-width / 2, "wrapper", "#2a6f97"), (width / 2, "all-features baseline", "#c9a227")):
    means, errors = [], []
    for model_name in model_order:
        values = results_cv[(results_cv["model"] == model_name) & (results_cv["variant"] == variant)][
            "outer_macro_f1"
        ].to_numpy()
        means.append(values.mean())
        errors.append(values.std(ddof=1) / np.sqrt(len(values)) if len(values) > 1 else 0.0)
    axes[0].bar(positions + offset, means, width, yerr=errors, capsize=3, label=variant, color=colour, alpha=0.9)

axes[0].axhline(chance_accuracy, color="crimson", linestyle="--", linewidth=1.2, label="chance level")
axes[0].set_xticks(positions)
axes[0].set_xticklabels(model_order, rotation=20, ha="right")
axes[0].set_ylabel("Outer cross-validation macro F1")
axes[0].set_title("Model comparison on the training portion\n(wrapper search vs all-features baseline)")
axes[0].legend(fontsize=8)

wrapper_only = results_cv[results_cv["variant"] == "wrapper"]
data_for_box = [
    wrapper_only[wrapper_only["model"] == model_name]["outer_macro_f1"].to_numpy()
    for model_name in model_order
]
box = axes[1].boxplot(data_for_box, labels=model_order, patch_artist=True, showmeans=True)
for patch in box["boxes"]:
    patch.set_facecolor("#9ecae1")
for mean_marker in box["means"]:
    mean_marker.set_markerfacecolor("#c1443c")
axes[1].axhline(chance_accuracy, color="crimson", linestyle="--", linewidth=1.2, label="chance level")
axes[1].set_ylabel("Outer cross-validation macro F1")
axes[1].set_title("Fold-to-fold variability of the wrapper variant\n(red marker = mean)")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

# ------------------------------------------------------------------ #
# 7.2  Wrapper subset heatmap
# ------------------------------------------------------------------ #
fig, ax = plt.subplots(figsize=(13, 4.6))
heatmap_values = subset_pivot.copy()
sns.heatmap(
    heatmap_values,
    annot=True,
    fmt=".3f",
    cmap="YlGnBu",
    ax=ax,
    linewidths=0.5,
    cbar_kws={"label": "mean inner-CV macro F1"},
)
ax.set_title("Wrapper search: mean inner-CV macro F1 for every feature subset")
ax.set_xlabel("Feature subset")
ax.set_ylabel("Model")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.tight_layout()
plt.show()

# ------------------------------------------------------------------ #
# 7.3  Selection stability
# ------------------------------------------------------------------ #
if not stability.empty:
    fig, ax = plt.subplots(figsize=(11, 4.6))
    pivot_stability = stability.pivot(index="model", columns="subset", values="share").fillna(0.0)
    pivot_stability = pivot_stability[[c for c in ordered_subsets if c in pivot_stability.columns]]
    pivot_stability.plot(kind="bar", stacked=True, ax=ax, colormap="tab20", edgecolor="white")
    ax.set_ylabel("Share of outer folds in which the subset was chosen")
    ax.set_xlabel("Model")
    ax.set_title("Wrapper selection stability across the outer folds")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7, ncol=2, title="Selected subset")
    plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
    fig.tight_layout()
    plt.show()

# ------------------------------------------------------------------ #
# 7.4  Learning curve of the selected model
# ------------------------------------------------------------------ #
banner("Learning curve of the selected model (training portion)")

learning_estimator = build_estimator(SELECTED_MODEL, FINAL_CONFIG)
# train_sizes are given as fractions of each cross-validation training fold,
# which is what scikit-learn expects; the axis is converted back to an
# approximate number of segments for readability.
train_size_fractions = np.linspace(0.25, 1.0, 5)
fold_training_size = len(X_train) * (N_SPLITS_OUTER - 1) / N_SPLITS_OUTER

returned_sizes, train_scores, validation_scores = learning_curve(
    learning_estimator,
    X_train[FINAL_COLUMNS],
    y_train,
    cv=StratifiedKFold(n_splits=N_SPLITS_OUTER, shuffle=True, random_state=RANDOM_STATE),
    train_sizes=train_size_fractions,
    scoring="f1_macro",
    n_jobs=1,
)
train_std = train_scores.std(axis=1)
validation_std = validation_scores.std(axis=1)
train_sizes = np.asarray(returned_sizes) * fold_training_size

fig, ax = plt.subplots(figsize=(9, 5.4))
ax.plot(train_sizes, train_scores.mean(axis=1), "o-", color="#c1443c", label="training score")
ax.fill_between(
    train_sizes,
    train_scores.mean(axis=1) - train_scores.std(axis=1),
    train_scores.mean(axis=1) + train_scores.std(axis=1),
    alpha=0.18,
    color="#c1443c",
)
ax.plot(train_sizes, validation_scores.mean(axis=1), "o-", color="#2a6f97", label="validation score")
ax.fill_between(
    train_sizes,
    validation_scores.mean(axis=1) - validation_scores.std(axis=1),
    validation_scores.mean(axis=1) + validation_scores.std(axis=1),
    alpha=0.18,
    color="#2a6f97",
)
ax.set_xlabel("Number of training samples")
ax.set_ylabel("Macro F1")
ax.set_title(
    f"Learning curve — {SELECTED_MODEL} on {', '.join(FINAL_COLUMNS)}\n"
    "(training portion only; the test portion is not involved)"
)
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

final_gap = float(train_scores.mean(axis=1)[-1] - validation_scores.mean(axis=1)[-1])
slope = float(validation_scores.mean(axis=1)[-1] - validation_scores.mean(axis=1)[0])
print(f"Training macro F1 at full training size : {train_scores.mean(axis=1)[-1]:.4f}")
print(f"Validation macro F1 at full training size: {validation_scores.mean(axis=1)[-1]:.4f}")
print(f"Train-validation gap                    : {final_gap:.4f}")
print(f"Validation change from 20% to 100% data : {slope:+.4f}")
if final_gap > 0.15:
    print("  A gap this size indicates over-fitting: the model fits the training data")
    print("  noticeably better than it generalises. Constraining capacity or adding data")
    print("  would be the next move.")
elif slope > 0.05:
    print("  The validation score is still improving with more data, so the model is")
    print("  DATA-limited rather than capacity-limited. More segments would help.")
else:
    print("  The validation score has flattened: the model is at the ceiling that these")
    print("  three features allow, so more data of the same kind would not help much.")

# ------------------------------------------------------------------ #
# 7.5  Per-class performance
# ------------------------------------------------------------------ #
report = classification_report(y_test, test_predictions, output_dict=True, zero_division=0)
class_labels = sort_labels(y.unique())


def report_row(report_dict, label) -> dict:
    """Fetch a per-class row from a classification report.

    scikit-learn writes the dictionary keys of a numeric-label report as
    strings, so the lookup has to try both forms.
    """
    for key in (label, str(label)):
        try:
            if key in report_dict:
                return report_dict[key]
        except TypeError:
            continue
    raise KeyError(f"Label {label!r} is not present in the classification report.")


per_class = pd.DataFrame(
    {
        "precision": {label: report_row(report, label)["precision"] for label in class_labels},
        "recall": {label: report_row(report, label)["recall"] for label in class_labels},
        "f1": {label: report_row(report, label)["f1-score"] for label in class_labels},
        "support": {label: report_row(report, label)["support"] for label in class_labels},
    }
)
per_class = per_class.sort_values("f1")

fig, ax = plt.subplots(figsize=(10, 5.2))
index = np.arange(len(per_class))
ax.barh(index - 0.2, per_class["precision"], 0.4, label="precision", color="#2a6f97")
ax.barh(index + 0.2, per_class["recall"], 0.4, label="recall", color="#c9a227")
ax.set_yticks(index)
ax.set_yticklabels([str(label) for label in per_class.index])
ax.set_xlim(0, 1)
ax.set_xlabel("Score")
ax.set_title("Per-label precision and recall on the held-out test portion")
ax.legend()
fig.tight_layout()
plt.show()

print("Per-label scores on the test portion")
print(per_class.round(4).to_string())

# ------------------------------------------------------------------ #
# 7.6  Permutation importance of the selected features
# ------------------------------------------------------------------ #
banner("Permutation importance (test portion)")

permutation = permutation_importance(
    final_estimator,
    X_test[FINAL_COLUMNS],
    y_test,
    n_repeats=30,
    random_state=RANDOM_STATE,
    scoring="f1_macro",
    n_jobs=1,
)
importance_frame = pd.DataFrame(
    {
        "feature": FINAL_COLUMNS,
        "importance_mean": permutation.importances_mean,
        "importance_std": permutation.importances_std,
    }
).sort_values("importance_mean", ascending=False)

print(importance_frame.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(
    importance_frame["feature"],
    importance_frame["importance_mean"],
    xerr=importance_frame["importance_std"],
    color="#2a6f97",
    capsize=4,
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Decrease in macro F1 when the feature is permuted")
ax.set_title("Permutation importance of the selected features")
fig.tight_layout()
plt.show()

print(
    "A feature whose permutation barely changes the score contributes little to THIS model.\n"
    "That is a statement about this model on this data, not a general claim about the\n"
    "feature — the wrapper decision in Sprint 5 was made on the training portion alone."
)

---
## Sprint 8 — Retrospective, limitations and the discussion the author must write

**Goal.** Close the loop: state what the run actually showed, record the methodological decisions, and be explicit about what this experiment cannot support.

### 8.1 Retrospective against the Sprint 0 contract

| Acceptance criterion (Sprint 0) | Where it is satisfied | Status |
| --- | --- | --- |
| Executable top-to-bottom in Colab after the data path is supplied | Sprint 1 configuration cell | Verify on the real data |
| Data-quality summary, class counts, train/test sizes and removed rows printed | Sprint 1 audit, Sprint 2 cleaning report | Automatic |
| All models compared under an identical split, folds and scoring | Sprint 5, shared `outer_cv` / `inner_cv` | Automatic |
| Wrapper compared against a no-selection baseline under the same protocol | Sprint 5 (nested CV) and Sprint 6 (test portion) | Automatic |
| Macro metrics and a confusion matrix reported; no single FPR for a multi-class task | Sprint 6 | Automatic |
| Over-fitting diagnosed with learning curves and train/validation gaps | Sprint 7 | Automatic |
| No invented numbers in any Markdown cell | Every figure and table is produced by code | Automatic |
| Discussion completed with the observed values | **Section 8.3 — written by the human author** | **Manual** |

### 8.2 The four methodological questions this project had to answer

| Question | Answer produced by this notebook |
| --- | --- |
| Which of the three features actually separate the users? | The wrapper subset table (Sprint 5) and the permutation importances (Sprint 7) |
| Which model generalises best under a fair protocol? | The nested cross-validation comparison and the one-standard-error rule (Sprint 5) |
| How good is the result on data the model has never seen? | The single held-out test evaluation with a bootstrap confidence interval (Sprint 6) |
| Can the model's confidence be trusted? | The reliability diagram, ECE, Brier score and confidence distribution (Sprint 6) |

### 8.3 Limitations that must be stated in the written discussion

1. **Segment-level random splitting.** Rows are individual movement segments, so segments from the same recording session can appear in both the training and the test portion. This inflates the apparent difficulty of the task relative to a genuine new session, because the model may recognise the *session* rather than the *user*. A session- or participant-level split would be the correct protocol, and it will be stricter. No session identifier is available in the delivered data, and none is invented here.
2. **Three features is a hard ceiling.** Distance, speed and acceleration are three summaries of one movement event. They omit direction, curvature, timing structure, pause behaviour, and device geometry. The result therefore describes the ceiling of *this* feature set, not the ceiling of mouse-dynamics biometrics.
3. **Small sample size and few users.** With roughly ten labels and a few thousand segments, a single mis-segmented file or one unusual recording can move macro F1 by several points. The bootstrap interval in Sprint 6 quantifies this; the fold-to-fold spread in Sprint 5 quantifies it again.
4. **Device, screen geometry and task context** are uncontrolled and are plausible confounders: a different laptop or a different multiple-choice task could shift the feature distributions more than the difference between two users.
5. **This is identification, not authentication.** The multi-class framing used here is not the framing an online-assessment proctoring system would use, and a single false-positive rate is not an appropriate summary for it. A verification protocol (one claimed user versus impostors) with FAR/FRR and an operating threshold is a different experiment with a different protocol, and it is deliberately not attempted here.
6. **No claim of deployment readiness.** The experiment shows what three movement features can do on this dataset. It does not show that the method detects cheating, and it must not be presented as a fraud-detection result.

### 8.4 What the human author must do before submission

* [ ] Run the notebook on the **real** course data with `USE_SYNTHETIC_DEMO = False` and `DATA_DIR` pointing at the mounted folder.
* [ ] Copy the **actual** numbers from Sprint 5, Sprint 6 and Sprint 7 into the project README result table; do not retype them from memory and do not keep the placeholder values.
* [ ] Replace every `<FILL IN ...>` placeholder in the discussion with an observation that the corresponding output supports, or remove the claim.
* [ ] If the wrapper did **not** beat the all-features baseline, say so and discuss why (redundancy, noise, or a genuinely small pool).
* [ ] State the limitation that applies to the result actually obtained — for example, the case in which the bootstrap interval overlaps chance level.
* [ ] Keep the data out of any public repository, and follow the course and institutional requirements for behavioural biometric data.

In [ ]:
"""Sprint 8 (implementation) — auto-generated results summary for the README and the discussion.

Everything printed below is derived from the values computed in the earlier
cells. It contains no hand-written performance claim, so it cannot drift
away from what the notebook actually produced.
"""

banner("Sprint 8 — auto-generated summary (copy this into the project README)")

if data_source != "course files":
    print(
        "WARNING: this run used the SYNTHETIC smoke-test data. The summary below describes\n"
        "the generator, not the course dataset, and it must not be submitted.\n"
    )

summary_lines = [
    "## Results (auto-generated from the notebook run)",
    "",
    f"- **Data source:** {data_source}",
    f"- **Segments analysed:** {len(X):,} across {y.nunique()} user labels",
    f"- **Rows removed during cleaning:** {rows_removed:,} of {rows_before:,}",
    f"- **Split:** stratified 80/20, `random_state={RANDOM_STATE}` "
    f"({len(X_train):,} train / {len(X_test):,} test)",
    f"- **Selected model:** {SELECTED_MODEL}",
    f"- **Selected feature subset (wrapper):** {', '.join(FINAL_SUBSET)}",
    f"- **Selected parameters:** {FINAL_CONFIG}",
    "",
    "### Held-out test portion",
    "",
    "| Model | Accuracy | Macro precision | Macro recall | Macro F1 | 95% CI (macro F1) |",
    "| --- | ---: | ---: | ---: | ---: | --- |",
    (
        f"| {SELECTED_MODEL} (wrapper) | {test_metrics['accuracy']:.4f} | "
        f"{test_metrics['macro_precision']:.4f} | {test_metrics['macro_recall']:.4f} | "
        f"{test_metrics['macro_f1']:.4f} | [{lower:.4f}, {upper:.4f}] |"
    ),
    (
        f"| {SELECTED_MODEL} (all three features) | {baseline_metrics['accuracy']:.4f} | "
        f"{baseline_metrics['macro_precision']:.4f} | {baseline_metrics['macro_recall']:.4f} | "
        f"{baseline_metrics['macro_f1']:.4f} | not computed |"
    ),
    (
        f"| Majority-class dummy | {dummy_metrics['accuracy']:.4f} | "
        f"{dummy_metrics['macro_precision']:.4f} | {dummy_metrics['macro_recall']:.4f} | "
        f"{dummy_metrics['macro_f1']:.4f} | not computed |"
    ),
    f"| Chance level | {chance_accuracy:.4f} | — | — | — | — |",
    "",
    "### Nested cross-validation (training portion only)",
    "",
    "| Model | Wrapper macro F1 (mean ± s.e.) | All-features baseline macro F1 (mean ± s.e.) |",
    "| --- | ---: | ---: |",
]

for model_name in model_order:
    wrapper_values = wrapper_only[wrapper_only["model"] == model_name]["outer_macro_f1"].to_numpy()
    baseline_values = results_cv[
        (results_cv["model"] == model_name) & (results_cv["variant"] == "all-features baseline")
    ]["outer_macro_f1"].to_numpy()
    summary_lines.append(
        f"| {model_name} | {wrapper_values.mean():.4f} ± "
        f"{(wrapper_values.std(ddof=1) / np.sqrt(len(wrapper_values)) if len(wrapper_values) > 1 else 0):.4f} | "
        f"{baseline_values.mean():.4f} ± "
        f"{(baseline_values.std(ddof=1) / np.sqrt(len(baseline_values)) if len(baseline_values) > 1 else 0):.4f} |"
    )

summary_lines += [
    "",
    "### Confidence and calibration (test portion)",
    "",
    f"- Mean predicted confidence: **{mean_confidence:.4f}**; observed accuracy: **{observed_accuracy:.4f}**",
    f"- Expected calibration error: **{ece:.4f}**; multiclass Brier score: **{brier:.4f}**; log loss: **{logloss:.4f}**",
    f"- One-vs-rest ROC AUC — macro: **{macro_auc:.4f}**, weighted: **{weighted_auc:.4f}**",
    f"- Train/validation gap at full training size: **{final_gap:.4f}**",
    "",
    "### Wrapper selection stability",
    "",
    "| Model | Subset | Times chosen (of 5 outer folds) |",
    "| --- | --- | ---: |",
]
for _, row in stability.sort_values(["model", "times_chosen"], ascending=[True, False]).iterrows():
    summary_lines.append(f"| {row['model']} | {', '.join(row['subset'])} | {int(row['times_chosen'])} / {int(row['folds'])} |")

summary_lines += [
    "",
    "### Permutation importance of the selected features (test portion)",
    "",
    "| Feature | Decrease in macro F1 when permuted |",
    "| --- | ---: |",
]
for _, row in importance_frame.iterrows():
    summary_lines.append(f"| {row['feature']} | {row['importance_mean']:.4f} ± {row['importance_std']:.4f} |")

summary_lines += [
    "",
    "---",
    "",
    "**Note.** A single false-positive rate is deliberately not reported: this is a multi-class",
    "identification task, and the appropriate summary is the macro-averaged metrics above.",
    "A verification protocol with FAR/FRR would be a separate experiment.",
    "",
]

print("\n".join(summary_lines))

print("=" * 78)
print("Discussion placeholders — the human author completes these AFTER the run")
print("=" * 78)
placeholders = [
    "1. Which model achieved the highest observed macro F1, and was the margin over the "
    "runner-up larger than the fold-to-fold noise? (cite Sprint 5)",
    "2. Which feature subset did the wrapper select, and was that choice stable across the "
    "outer folds? (cite the stability table in Sprint 5)",
    "3. Did the wrapper beat the all-features baseline? If not, why not — and what does that "
    "say about redundancy between the three features? (cite Sprint 5 and Sprint 6)",
    "4. Which labels were most often confused, and is the pattern consistent with the "
    "per-label scores in Sprint 6 and the confusion matrix?",
    "5. What do the three features capture, and what do they omit?",
    "6. Is the model over-fitting, data-limited or at its ceiling? (cite the learning curve "
    "in Sprint 7)",
    "7. Is the model calibrated? What does the reliability diagram say about how far its "
    "confidence can be trusted? (cite Sprint 6)",
    "8. Which limitations in Section 8.3 apply to the result you actually obtained, and which "
    "do not?",
    "9. Restate the boundary: this is exploratory identification on three features, not a "
    "fraud-detection or authentication result.",
]
for item in placeholders:
    print(f"  <FILL IN AFTER RUNNING> {item}")